# **MÁSTER BASKET DATA ANALYTICS - CALCULADORA MEJORADA**



## **ASIGNATURA**: **Programación Avanzada**

## **ALUMNO**: **Pablo Gómez Villamayor**
## CURSO: 2025-2026


---
---
---
---
---



**SITUACIÓN HIPOTÉTICA** Soy un analista de datos en un club de baloncesto y tengo que programar una forma eficiente de introducir los datos de los jugadores de mi equipo en los partidos, para poder hacer obtener métricas y realizar análisis.

**PUNTO DE PARTIDA**: **Código de la anterior calculadora**. Hemos buscado **re-factorizarlo**, incluyendo nuevos conceptos aprendidos: Control de excepciones, logging, list comprehesions, iteradores zip(), tipos de atributos (posicionales, por defecto, *args y **kwargs) y programación orientada a objetos.

Tenemos varios **objetivos**: Por un lado, que el código sea limpio, mantenible y lo más completo posible. Y por otro: Implementar control de valores imputados (datos de entrada coherentes), usar funciones para ahorrar líneas de código, calcular mínimo 3 métricas avanzadas y comentar adecuadamente el código.

---
---

La **PRINCIPAL DIFERENCIA** entre esta calculadora y la anterior es el **ENFOQUE**:

- Enfoque anterior -> PROGRAMACIÓN PROCEDURAL (o procedimental).

- **Enfoque nuevo**    -> **PROGRAMACIÓN ORIENTADA A OBJETOS (o POO)**.

A diferencia de en la versión anterior, ahora no vamos a tener diferentes calculadoras, independientes, que van escalando complejidad. Vamos a realizar una arquitectura con diferentes clases, y calcularemos las métricas avanzadas con métodos, dentro de dichas clases.

---
---

## **EXPLICACIÓN DE LA ARQUITECTURA: ESTRUCTURA JERÁRQUICA**


Season

│

├── Game (Jornada)

│   │

│   ├── MyTeamGame

│   │   ├── PlayerGameStats

│   │   │   └── Player

│   │   │

│   └── OpponentTeamGame

│

└── Players (Registro general)

---
---


-Relaciones principales:

- 'Player' es la entidad del jugador -> Un player participa en múltiples partidos.
- 'PlayerGameStats' representa las estadísticas de un jugador en un partido.
- 'TeamGameStats' representa las estadísticas de mi equipo que no se asocian a ningún jugador (rebotes, pérdidas, faltás técnicas...).
- 'MyTeamGame' representa todas las estadísticas de mi equipo en el partido: jugadores + equipo + totales.
- 'OpponentTeamGame' representa las estadísticas totales del equipo rival en el partido.
- 'Game' contiene toda la información del partido/jornada -> i.e. agrega estadísticas de ambos equipos (MyTeamGame & OpponentTeamGame).

Finalmente:

- 'Season' agrega múltiples partidos de mi equipo -> CALCULADORA FINAL.

Sin más explicaciones, vamos con el código.

**=============================================================**
## **CAPA 0 — IMPORTS Y CONFIGURACIÓN: LOGGING**
**=============================================================**

In [236]:
import json
import logging
from dataclasses import dataclass, field
from typing import List, Dict, Optional

In [237]:
import pandas as pd
pd.set_option('display.precision', 2) # Para que muestre los datos con dos decimales.
import numpy as np

In [238]:
# Limpiar handlers anteriores (Colab)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configuración del sistema de logs
logging.basicConfig(
    #filename="season_calculator.log",
    level=logging.INFO, # Nivel mínimo que se mostrará
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    handlers=[
        logging.FileHandler("LOGS_TFA_PGV_ProgramacionAvanzada.log"),   # Archivo
        logging.StreamHandler()                   # Consola
    ]
)

logger = logging.getLogger(__name__)



In [239]:
import datetime

In [240]:
import re

# 're' es el módulo de expresiones regulares en Python.
# Estas expresiones son patrones que permiten buscar o validar cadenas de texto de manera muy flexible.


**============================================================**
## **CAPA 1 — UTILIDADES:**
## **Excepciones, Funciones, Métricas & Calculadoras**
**============================================================**

**MIS EXCEPCIONES**

In [241]:
# ==============================
# EXCEPCIONES PERSONALIZADAS
# ==============================

class ValidationError(Exception):
    """Excepción base para validaciones."""
    pass

class NombreInvalidoError(ValidationError):
    """Error en validación de nombre."""
    pass

class DorsalInvalidoError(ValidationError):
    """Error en validación de dorsal."""
    pass

class DorsalDuplicadoEnPartidoError(ValidationError):
    """Se produce cuando dos jugadores comparten dorsal en el mismo partido."""
    pass

class NumeroJugadoresError(ValidationError):
    """Error en número de jugadores en partido. Deben jugar al menos 5 jugadores."""
    pass

class PlayerYStatsNoCorrespondientesError(ValidationError):
    """Las estadísticas que quieren añadirse no corresponden a ese jugador."""
    pass

**MIS FUNCIONES**

In [242]:
def add_game_to_players(players_list, stats_list):
    """
    Añade las estadísticas de un partido a cada jugador correspondiente.
    Ambas listas deben estar alineadas por posición. En caso contrario, dará un error.
    """
    if len(players_list) != len(stats_list):
        raise ValueError("El número de jugadores y estadísticas no coincide.")

    for player, stats in zip(players_list, stats_list):
        if stats.player is not player:
            raise PlayerYStatsNoCorrespondientesError(f"Hay un problema. Revisa la lista de tus jugadores y la lista de sus estadísticas. \nComprueba que ambas listas coinciden en el orden adecuado. \nParece que las estadísticas de {player.name} no se han añadido correctamente.")
        player.add_game(stats)


In [243]:
def to_dataframe(esto):
    """Convierte un diccionario en DataFrame."""
    return pd.DataFrame([esto])

In [244]:
def format_minutes(min_decimal):

    if min_decimal == "" or pd.isna(min_decimal):
        return ""

    total_seconds = int(min_decimal * 60)
    minutes = total_seconds // 60
    seconds = total_seconds % 60

    return f"{minutes}:{seconds:02d}"

**MIS MÉTRICAS**

Las escribo como funciones globales. Esto permite calcularlas tanto para jugadores como para el equipo, sin necesidad de duplicar código.

Si no lo hiciéramos así habría que escribirlas como métodos en varias clases diferentes, aumentando la longitud total del código.

In [339]:
# ==========================================================
# ESTADÍSTICA CONVENCIONAL
# ==========================================================

def min_played(minutos, segundos):
    """Minutos totales jugados"""
    return (minutos + (segundos/60))

def tl_pct(tl_anot, tl_int):
    """ % Tiros libres (FT) """
    return ((tl_anot / tl_int) * 100) if tl_int > 0 else 0

def t2_pct(t2_anot, t2_int):
    """ % Tiros de 2 puntos """
    return ((t2_anot / t2_int) * 100) if t2_int > 0 else 0

def t3_pct(t3_anot, t3_int):
    """ % Tiros de 3 puntos """
    return ((t3_anot / t3_int) * 100) if t3_int > 0 else 0

def fg_anot(t2_anot, t3_anot):
    """ Tiros de campo anotados """
    return (t2_anot + t3_anot)

def fg_int(t2_int, t3_int):
    """ Tiros de campo intentados """
    return (t2_int + t3_int)

def fg_pct(fg_anot, fg_int):
    """ % Tiros de campo """
    return ((fg_anot / fg_int) * 100) if fg_int > 0 else 0

def reb_tot(reb_o, reb_d):
    """ Rebotes totales"""
    return (reb_o + reb_d)

def missed_shots(tl_anot, tl_int, t2_anot, t2_int, t3_anot, t3_int):
    """ Tiros fallados """
    return ((tl_int - tl_anot) + (t2_int - t2_anot) +(t3_int - t3_anot))

def valoracion_acb(puntos, reb_o, reb_d, ast, rec, tap_f, fr, tl_anot, tl_int, t2_anot, t2_int, t3_anot, t3_int, perd, tap_c, fc):
    """ Valoración ACB"""
    return ((puntos + reb_tot(reb_o, reb_d) + ast + rec + tap_f + fr) - (missed_shots(tl_anot, tl_int, t2_anot, t2_int, t3_anot, t3_int) + perd + tap_c + fc))

# ===============================================================
# ESTADÍSTICA AVANZADA: MÉTRICAS DE JUGADORES & EQUIPO - Parte 1
# ===============================================================

def posesiones_consumidas(tl_int, t2_int, t3_int, perd):
    """ Posesiones consumidas por un jugador o equipo. NO contabiliza rebote ofensivo """
    return (fg_int(t2_int, t3_int) + 0.44 * tl_int + perd)

def posesiones(tl_int, t2_int, t3_int, reb_o, perd):
    """ Posesiones consumidas por el equipo. SÍ contabiliza rebote ofensivo """
    return (fg_int(t2_int, t3_int) + 0.44 * tl_int - reb_o + perd)

def oer(puntos, posesiones):
    """ Offensive Efficiency Rating - OER -> EQUIPO: POR 100 POSESIONES"""
    return ((puntos / posesiones)*100) if posesiones > 0 else 0

def oer_player(puntos, posesiones):
    """ Offensive Efficiency Rating - OER -> JUGADOR: POR 1 POSESIÓN """
    return (puntos / posesiones) if posesiones > 0 else 0

# ===============================================================
# ESTADÍSTICA AVANZADA: MÉTRICAS DE JUGADORES & EQUIPO - Parte 2

# ------------------- 4 FACTORS (Dean Oliver) -------------------
# ===============================================================


def efg(t2_anot, t2_int, t3_anot, t3_int):
    """ Effective Field Goal % """
    return ( ( (fg_anot(t2_anot, t3_anot) + 0.5 * t3_anot) / fg_int(t2_int, t3_int) ) * 100 ) if fg_int(t2_int, t3_int) > 0 else 0

# %PÉRDIDAS -> NO CONTABILIZAMOS REBOTE OFENSIVO. VER https://hackastat.eu/en/learn-a-stat-assists-and-turnovers/
def tov_ratio_team(perdidas, tl_int, t2_int, t3_int, perd):
    """ % de pérdidas cometidas - EQUIPO """
    denom = posesiones_consumidas(tl_int, t2_int, t3_int, perd)
    return ( (perdidas / denom ) * 100 ) if denom > 0 else 0

def tov_ratio_player(tl_int, t2_int, t3_int, perd, ast):
      """ % de pérdidas cometidas - JUGADOR """
      denom = posesiones_consumidas(tl_int, t2_int, t3_int, perd) + ast
      return ( (perd / denom ) * 100 ) if denom > 0 else 0

def ft_rate(tl_int, t2_int, t3_int):
    """ Ratio TL-Tiro Campo """
    return  ( tl_int / fg_int(t2_int,t3_int) )  if fg_int(t2_int,t3_int) > 0 else 0

#%REBOTES -> https://hackastat.eu/en/learn-a-stat-rebound-percentages/
def oreb_pct(oreb_team, dreb_opp):
    """ % de rebote ofensivo - EQUIPO """
    return (( oreb_team / (oreb_team + dreb_opp) ) * 100 ) if (oreb_team + dreb_opp) > 0 else 0


# % REBOTE OFENSIVO -> VERSIÓN PARA JUGADORES. Pondera por tiempo jugado
def oreb_pct_player(oreb_player, min_player, oreb_team, min_team, dreb_opp):
    """ % de rebote ofensivo - JUGADOR"""
    rebotes_disponibles = (oreb_team + dreb_opp)
    factor = (min_player / (min_team / 5)) # Fracción del tiempo en pista del jugador.
                                           # Es un factor entre 0 y 1 (0 si no juega nada. 1 si juega todos los minutos )
    denom = rebotes_disponibles * factor
    return ( (oreb_player / denom) * 100 ) if denom > 0 else 0

# ===============================================================
# ESTADÍSTICA AVANZADA: MÉTRICAS DE JUGADORES & EQUIPO - Parte 3
# ===============================================================

def dreb_pct(dreb_team, oreb_opp):
    """ % de rebote defensivo """
    return (( dreb_team / (dreb_team + oreb_opp) ) * 100 ) if (dreb_team + oreb_opp) > 0 else 0

# % REBOTE DEFENSIVO -> VERSIÓN PARA JUGADORES. Pondera por tiempo jugado
def dreb_pct_player(dreb_player, min_player, dreb_team, min_team, oreb_opp):
    """ % de rebote defensivo - JUGADOR"""
    rebotes_disponibles = (dreb_team + oreb_opp)
    factor = (min_player / (min_team / 5))
    denom = rebotes_disponibles * factor
    return ( (dreb_player / denom) * 100 ) if denom > 0 else 0


def ts(puntos, tl_int, t2_int, t3_int):
    """ True Shooting % """
    denom = fg_int(t2_int, t3_int) + 0.44 * tl_int
    return ((puntos * 0.5 / denom) * 100) if denom > 0 else 0

def ast_tov_ratio(ast, perd):
    """ Ratio Asistencia-Pérdida """
    return (ast / perd) if perd > 0 else ast

def ast_ratio_team(ast, tl_int, t2_int, t3_int, perd):
    """ % de las posesiones del EQUIPO que acaban con una asistencia """
    denom = posesiones_consumidas(tl_int, t2_int, t3_int, perd)
    return ((ast / denom ) *100) if denom > 0 else 0

def ast_ratio_player(ast, tl_int, t2_int, t3_int, perd):
    """ % de las posesiones consumidas por un JUGADOR que acaban con una asistencia """
    denom = posesiones_consumidas(tl_int, t2_int, t3_int, perd) + ast
    return ((ast / denom ) *100) if denom > 0 else 0

def ast_pct_team(ast, t2_int, t3_int):
    """ % de tiros del equipo que son asistidos.
    Se obtiene al dividir las asistencias entre todos los tiros de campo (2pt y 3pt) intentados."""
    return  ( ( ast / fg_int(t2_int,t3_int) ) * 100 ) if fg_int(t2_int,t3_int) > 0 else 0

def ast_pct_player(ast, t2_int_player, t3_int_player, min_player, t2_int_team, t3_int_team, min_team):
    """ % de tiros asistidos por un jugador durante su tiempo en cancha.
        Tiene en cuenta todos los tiros que potencialmente podía asistir mientras está en la pista,
        por eso se restan los tiros intentados por el jugador. Métrica aproximada, pues pondera por
        tiempo jugado del partido. No en todos los momentos del partido se tiran el mismo númro de tiros. """
    factor = (min_player / (min_team / 5))
    return  ( ( ast / ( fg_int(t2_int_team,t3_int_team) * factor - fg_int(t2_int_player,t3_int_player) ) ) * 100 )

# %ROBOS Y %TAPONES -> https://hackastat.eu/en/learn-a-stat-steal-and-block-percentage/
def stl_pct(robos_equipo, tl_int_opp, t2_int_opp, t3_int_opp, oreb_opp, perd_opp):
    """ % de robos conseguidos - EQUIPO """
    posesiones_opp = posesiones(tl_int_opp,t2_int_opp,t3_int_opp,oreb_opp,perd_opp)
    return ( (robos_equipo / posesiones_opp) * 100 ) if posesiones_opp > 0 else 0

def stl_pct_player(robos_player, min_player, min_team, tl_int_opp,t2_int_opp,t3_int_opp,oreb_opp,perd_opp):
    """ % de robos conseguidos - JUGADOR """
    posesiones_opp = posesiones(tl_int_opp,t2_int_opp,t3_int_opp,oreb_opp,perd_opp)
    factor = (min_player / (min_team / 5))
    denom = posesiones_opp * factor
    return ( (robos_player / denom ) * 100 ) if denom > 0 else 0

#La estadística de BLOCK%, según la web consultada, no contabiliza posibles tapones en triples.
#Con la evolución del juego actual, podríamos tenerlo en cuenta. De momento, NO lo hacemos.

def blk_pct(tapones_equipo, t2_int_rival):
    """ % de tapones conseguidos - EQUIPO """
    return ( (tapones_equipo / t2_int_rival) * 100 ) if t2_int_rival > 0 else 0

def blk_pct_player(tapones_player, t2_int_rival, min_player, min_team):
    """ % de tapones conseguidos - JUGADOR """
    factor = (min_player / (min_team / 5))
    denom = t2_int_rival * factor
    return ( (tapones_player / denom) * 100 ) if denom > 0 else 0


# ==========================================================
# ESTADÍSTICA AVANZADA: USAGE

# USG% & USG%_TOT
# Necesitan datos del equipo para calcularse: Minutos y posesiones totales.
# ==========================================================


# USG% (Uso durante tiempo en pista)
def usg(tl_int, t2_int, t3_int, perd, min_played, team_min, team_poss):
    """ % de uso de un jugador durante su tiempo en pista """
    factor = (min_played / (team_min / 5)) # Fracción del tiempo en pista del jugador.
    return ( (fg_int(t2_int, t3_int) + 0.44 * tl_int + perd) / (team_poss * factor)  * 100 ) if ((team_poss * factor) > 0)  else 0

# USG%_TOTAL (Uso respecto al total del equipo en el partido)
def usg_tot(tl_int, t2_int, t3_int, perd, team_poss):
    """ % de uso de un jugador respecto al total del equipo. Contabiliza todo el partido, no solo su tiempo en pista """
    return ( ( ( fg_int(t2_int, t3_int) + 0.44 * tl_int + perd) / team_poss ) * 100 ) if team_poss > 0 else 0


# ==========================================================
# ESTADÍSTICA AVANZADA: MÉTRICAS EXCLUSIVAS DE EQUIPO
# ==========================================================

def der(puntos_rival, posesiones_rival):
    """ Deffensive Efficiency Rating - DER """
    return (puntos_rival / posesiones_rival) * 100 if posesiones_rival > 0 else 0

def net_rating(off_rating, def_rating):
    """ Net Rating = 0ER - DER """
    return off_rating - def_rating


# PACE = Número de posesiones que juega un equipo por tiempo de juego reglamentario
# (ACB: 40 min/partido | NBA: 48 min/partido).
# Hay que ponderar la media de posesiones por tiempo de juego ->
# Si hay prórrogas, aumenta el número de posesiones, pero no el ritmo de juego.

def pace_acb(team_poss, opp_poss, team_min,T=200): # 200 = 40 min/jugador * 5 jugadores -> Minutos de equipo en ACB, SIN PRRÓRROGAAS.
    """ PACE: ~Ritmo de juego"""
    return ( (team_poss + opp_poss) / 2 ) * ( T / team_min)

def pace_nba(team_poss, opp_poss, team_min,T=240): # 240 = 48 min/jugador * 5 jugadores -> Minutos de equipo en NBA, SIN PRRÓRROGAAS.
    """ PACE: ~Ritmo de juego"""
    return ( (team_poss + opp_poss) / 2 ) * ( T / team_min)



#### **COMENTARIOS SOBRE ALGUNAS ESTADÍSTICAS AVANZADAS (+ ENLACES DE INTERÉS)**

Hay varias estadísticas avanzadas que se calculan de manera diferente para el equipo y para un jugador. ¡Hay que tener mucho cuidado con esto!

A continuación, incluimos enlaces a artículos/webs interesantes, en relación a varias de estas métricas:

---
---


-"OREB%, DREB%": Porcentaje de rebote (ofensivo y defensivo). Versiones de equipo y de jugadores. Se recomienda consultar:

https://hackastat.eu/en/learn-a-stat-rebound-percentages/

---

-"AST_RATIO" Y "AST%". Son métricas diferentes. Consúltese:

https://hackastat.eu/en/learn-a-stat-assists-and-turnovers/

---

-"STL%" Y "BLK%", puede consultarse:

https://hackastat.eu/en/learn-a-stat-steal-and-block-percentage/

---

-"PACE". Se recomienda consultar:

https://dsbasketball.es/index.php/glosario/2-pace-ritmo





#### **MIS CALCULADORAS**

Calculadoras de estadística avanzada, a partir de DataFrame:

- CALCULADORA 1: ESTADÍSTICA AVANZADA - JUGADORES.
- CALCULADORA 2: ESTADÍSTICA AVANZADA - EQUIPO.



In [337]:
# ==========================================================
# CALCULADORA 1: ADVANCED STATS — PLAYERS
# ==========================================================

def calculate_advanced_players(players_df, team_totals, opponent_totals):

    df = players_df.copy()
    team_poss = posesiones(team_totals["tl_int"],team_totals["t2_int"],team_totals["t3_int"],team_totals["reb_o"],team_totals["perd"])
    team_min = team_totals["min_played"]
    opp = opponent_totals

    # ----------------------------
    # POSESIONES CONSUMIDAS
    # ----------------------------

    df["POSS_USED"] = df.apply(lambda r: posesiones_consumidas(r.tl_int,r.t2_int,r.t3_int,r.perd),axis=1)

    # ----------------------------
    # OER
    # ----------------------------

    df["OER"] = df.apply(lambda r: oer_player(r.puntos, r.POSS_USED),axis=1)

    # ---------------------------------------
    # 4 FACTORS: eFG%, TOV%, OREB%, FT_RATE.
    # ---------------------------------------

    df["EFG%"] = df.apply(lambda r: efg(r.t2_anot, r.t2_int, r.t3_anot, r.t3_int),axis=1)
    df["TOV%"] = df.apply(lambda r: tov_ratio_player(r.tl_int,r.t2_int,r.t3_int,r.perd,r.ast),axis=1)
    df["OREB%"] = df.apply(lambda r: oreb_pct_player(r.reb_o,r.min_played,team_totals["reb_o"],team_min,opp["reb_d"]),axis=1)
    df["FT_RATE"] = df.apply(lambda r: ft_rate(r.tl_int,r.t2_int,r.t3_int),axis=1)

    # ----------------------------
    # DREB%
    # ----------------------------

    df["DREB%"] = df.apply(lambda r: dreb_pct_player(r.reb_d,r.min_played,team_totals["reb_d"],team_min,opp["reb_o"]),axis=1)

    # ----------------------------
    # TS%
    # ----------------------------

    df["TS%"] = df.apply(lambda r: ts(r.puntos,r.tl_int,r.t2_int,r.t3_int),axis=1)

    # ----------------------------
    # AST/TOV
    # ----------------------------

    df["AST/TOV"] = df.apply(lambda r: ast_tov_ratio(r.ast, r.perd),axis=1)

    # ----------------------------
    # AST_RATIO
    # ----------------------------

    df["AST_RATIO%"] = df.apply(lambda r: ast_ratio_player(r.ast,r.tl_int,r.t2_int,r.t3_int,r.perd),axis=1)

    # ----------------------------
    # AST%
    # ----------------------------

    df["AST%"] = df.apply(lambda r: ast_pct_player(r.ast,r.t2_int,r.t3_int,r.min_played,team_totals["t2_int"],team_totals["t3_int"],team_min),axis=1)

    # ----------------------------
    # STL%
    # ----------------------------

    df["STL%"] = df.apply(lambda r: stl_pct_player(r.rec,r.min_played,team_min,opp["tl_int"],opp["t2_int"],opp["t3_int"],opp["reb_o"],opp["perd"]),axis=1)

    # ----------------------------
    # BLK%
    # ----------------------------

    df["BLK%"] = df.apply(lambda r: blk_pct_player(r.tap_f,opp["t2_int"],r.min_played,team_min),axis=1)

    # ----------------------------
    # USAGE
    # ----------------------------

    team_poss_without_oreb = posesiones_consumidas(team_totals["tl_int"],team_totals["t2_int"],team_totals["t3_int"],team_totals["perd"])
    df["USG%"] = df.apply(lambda r: usg(r.tl_int,r.t2_int,r.t3_int,r.perd,r.min_played,team_min,team_poss_without_oreb),axis=1)
    df["USG_TOT%"] = df.apply(lambda r: usg_tot(r.tl_int,r.t2_int,r.t3_int,r.perd,team_poss_without_oreb),axis=1)

    # Para que la suma de "USG_TOT%" de todos los jugadores sea ~100%
    # (Hay un pequeño margen, no es exactamente 100% por las posesiones que se contabilizan en la fila de "Equipo")
    # El USAGE (USG% y USG_TOT%) da cuenta de las posesiones que consume un jugador respecto al total del equipo. ->
    # No debemos contar los rebotes ofensivos en las posesiones consumidas ni por el jugador, ni por el equipo.
    # La posesión no acaba nunca con rebote ofensivo: Lo hará con tiro, pérdida, etc

    # MOSTRAR RESULTADOS -> Nos quedamos solo con las columnas de estadística avanzada.
    columns_advanced=["jugador","dorsal","POSS_USED","OER","EFG%","TOV%","OREB%","FT_RATE","DREB%","TS%","AST/TOV","AST_RATIO%","AST%","STL%","BLK%","USG%","USG_TOT%"]
    df = df[columns_advanced]

    # Calculamos fila de totales, solo para las métricas que tiene sentido sumar.
    # El USG_TOT% debería sumar 100% | PERO EL USG% NO.

    df.loc[''] = df[["jugador","POSS_USED","USG%","USG_TOT%"]].sum()
    df.iat[-1, 0] = 'TOTALES'
    df_limpio=df.fillna("") #Eliminamos los "NaN"

    return df_limpio.round(2)

    #return df.round(2)

In [247]:
# ==========================================================
# CALCULADORA 2: ADVANCED STATS — TEAM
# ==========================================================

def calculate_advanced_team(team, opp):
    stats = {}
    team_poss = posesiones(team["tl_int"], team["t2_int"], team["t3_int"], team["reb_o"], team["perd"])
    opp_poss = posesiones(opp["tl_int"], opp["t2_int"], opp["t3_int"], opp["reb_o"], opp["perd"])
    stats["jugador"] = team["jugador"]
    stats["POSS"] = team_poss
    stats["OER"] = oer(team["puntos"], team_poss)
    stats["DER"] = der(opp["puntos"], opp_poss)
    stats["NET"] = net_rating(stats["OER"], stats["DER"])
    stats["EFG%"] = efg(team["t2_anot"], team["t2_int"], team["t3_anot"], team["t3_int"])
    stats["TOV%"] = tov_ratio_team(team["perd"], team["tl_int"], team["t2_int"], team["t3_int"], team["perd"])
    stats["OREB%"] = oreb_pct(team["reb_o"], opp["reb_d"])
    stats["FT_RATE"] = ft_rate(team["tl_int"], team["t2_int"], team["t3_int"])
    stats["DREB%"] = dreb_pct(team["reb_d"], opp["reb_o"])
    stats["TS%"] = ts(team["puntos"], team["tl_int"], team["t2_int"], team["t3_int"])
    stats["AST/TOV"] = ast_tov_ratio(team["ast"], team["perd"])
    stats["AST_RATIO%"] = ast_ratio_team(team["ast"], team["tl_int"], team["t2_int"], team["t3_int"], team["perd"])
    stats["AST%"] = ast_pct_team(team["ast"], team["t2_int"], team["t3_int"])
    stats["STL%"] = stl_pct(team["rec"], opp["tl_int"], opp["t2_int"], opp["t3_int"], opp["reb_o"], opp["perd"])
    stats["BLK%"] = blk_pct(team["tap_f"], opp["t2_int"])
    stats["PACE_ACB"] = pace_acb(team_poss, opp_poss, team["min_played"])
    stats["PACE_NBA"] = pace_nba(team_poss, opp_poss, team["min_played"])

    df = pd.DataFrame([stats]).round(2)

    return df


**=====================================**
## **CAPA 2 — DOMINIO**
**=====================================**


### **CLASE 'PlayerValidator'**

Asegura que el nombre y el dorsal de los jugadores son correctos.

Lo creamos a parte, para limpiar código de la clase 'Player'

In [248]:
# ==============================
# MÓDULO DE VALIDACIONES
# ==============================

class PlayerValidator:

    # ==============================
    # VALIDACIÓN DE NOMBRE
    # ==============================

    @staticmethod
    def validate_name(nombre: str) -> str:
        """
        Valida el nombre del jugador.

        Permite:
        - Mayúsculas internas (McCollum, LeBron)
        - Apóstrofes (O'Neal)
        - Puntos (O.J.)
        - Guiones (Gilgeous-Alexander)
        - Letras con acento (Dončić)
        """

        nombre = nombre.strip()
        nombre = " ".join(nombre.split())  # Elimina espacios duplicados

        # 1. No vacío
        if nombre == "":
            logger.error("Nombre inválido: vacío.")
            raise NombreInvalidoError("El nombre no puede estar vacío.")

        # 2. No empezar ni terminar con símbolo
        if nombre[0] in "-'." or nombre[-1] in "-'.":
            logger.error(f"Nombre inválido '{nombre}': empieza o termina con símbolo.")
            raise NombreInvalidoError("El nombre no puede empezar ni acabar con símbolos.")

        # 3. Caracteres permitidos
        if any(not (c.isalpha() or c in " .'-") for c in nombre):
            logger.error(f"Nombre inválido '{nombre}': contiene caracteres no permitidos.")
            raise NombreInvalidoError("El nombre contiene caracteres no válidos.")

        # 4. Secuencias inválidas
        invalidos = ["--","''", "..","- ", " -","' ", " '","-'", "'-","-.", ".-","'.", ".'"]

        if any(x in nombre for x in invalidos):
            logger.error(f"Nombre inválido '{nombre}': secuencia inválida de símbolos.")
            raise NombreInvalidoError("Uso incorrecto de guiones, apóstrofes o puntos.")

        return nombre


    # ==============================
    # VALIDACIÓN DE DORSAL
    # ==============================

    @staticmethod
    def validate_dorsal(dorsal: str) -> str:
        """
        Valida dorsal según reglamento ACB.
        Permite '00' y del '0' al '99'.
        Devuelve dorsal como STRING.
        """

        dorsal = dorsal.strip()

        # Solo dígitos
        if not dorsal.isdigit():
            logger.error(f"Dorsal inválido '{dorsal}': contiene caracteres no numéricos.")
            raise DorsalInvalidoError("El dorsal debe contener solo números.")

        # Longitud máxima 2
        if len(dorsal) > 2:
            logger.error(f"Dorsal inválido '{dorsal}': tiene más de dos cifras.")
            raise DorsalInvalidoError("El dorsal debe tener máximo dos cifras.")

        # Rango válido
        valor = int(dorsal)
        if valor < 0 or valor > 99:
            logger.error(f"Dorsal inválido '{dorsal}': fuera de rango 0-99.")
            raise DorsalInvalidoError("El dorsal debe estar entre 0 y 99.")

        # Caso especial permitido
        if dorsal in ("0", "00"):
            return dorsal

        # Evitar ceros a la izquierda
        if dorsal.startswith("0"):
            logger.error(f"Dorsal inválido '{dorsal}': cero a la izquierda no permitido.")
            raise DorsalInvalidoError("Puedes usar '0' o '00', pero no se permiten ceros a la izquierda.")

        return dorsal


### **CLASE 'Player' (Entidad permanente)**


In [249]:
@dataclass
class Player:
    """
    Representa un jugador a lo largo de la temporada.
    Guarda todas sus actuaciones (PlayerGameStats).
    """

    name: str
    dorsal: str
    games: List["PlayerGameStats"] = field(default_factory=list)
    #default_factory=list -> Cada vez que se cree un Player, crea una lista nueva e independiente para ese jugador.


    def __post_init__(self):
        # Validaciones automáticas al crear el jugador -> NOMBRE & DORSAL.
        self.name = PlayerValidator.validate_name(self.name)
        self.dorsal = PlayerValidator.validate_dorsal(self.dorsal)
        logger.info(f"Jugador creado: {self.name} (#{self.dorsal})")


    # ==============================
    # OTROS MÉTODOS
    # ==============================

    def add_game(self, game_stats):
        """
        Añade las estadísticas de un partido al jugador.
        """
        self.games.append(game_stats)

    def total_points_season(self):
        """
        Calcula los puntos totales acumulados en la temporada.
        """
        return sum(game.pts for game in self.games)

    def average_points_season(self):
        """
        Calcula la media de puntos por partido en la temporada.
        """
        if not self.games:
            return 0
        return self.total_points_season() / len(self.games)

    @property
    def games_played(self) -> int:
        return len(self.games)

Ahora, algunos ejemplos de creación de 'Players'

In [250]:
#EJEMPLO 1a: NOMBRE CORRECTO
player_2 = Player(name="Shai Gilgeous-Alexander", dorsal="2")
print(player_2)

2026-03-15 19:54:49,399 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)


Player(name='Shai Gilgeous-Alexander', dorsal='2', games=[])


In [251]:
#EJEMPLO 1b: NOMBRE CORRECTO
player_77 = Player(name="Luka Dončić", dorsal="77")
print(player_77)

2026-03-15 19:54:49,406 - INFO - __main__ - Jugador creado: Luka Dončić (#77)


Player(name='Luka Dončić', dorsal='77', games=[])


In [252]:
#EJEMPLO 2a: NOMBRE INCORRECTO
player_2 = Player(name="Shai Gilgeous--Alexander", dorsal="2")

2026-03-15 19:54:49,415 - ERROR - __main__ - Nombre inválido 'Shai Gilgeous--Alexander': secuencia inválida de símbolos.


NombreInvalidoError: Uso incorrecto de guiones, apóstrofes o puntos.

In [300]:
#EJEMPLO 2b: NOMBRE INCORRECTO
player_23 = Player(name="LeBron James.", dorsal="23")

2026-03-15 19:56:51,928 - ERROR - __main__ - Nombre inválido 'LeBron James.': empieza o termina con símbolo.


NombreInvalidoError: El nombre no puede empezar ni acabar con símbolos.

In [301]:
#EJEMPLO 3a: DORSAL CORRECTO
player_0 = Player(name="Markus Howard", dorsal="0")
print(player_0)

2026-03-15 19:56:54,682 - INFO - __main__ - Jugador creado: Markus Howard (#0)


Player(name='Markus Howard', dorsal='0', games=[])


In [302]:
#EJEMPLO 3b: DORSAL CORRECTO
player_00 = Player(name="Devin Robinson", dorsal="00")
print(player_00)

2026-03-15 19:56:56,732 - INFO - __main__ - Jugador creado: Devin Robinson (#00)


Player(name='Devin Robinson', dorsal='00', games=[])


In [303]:
#EJEMPLO 4a: DORSAL INCORRECTO
player_01 = Player(name="Markus Howard", dorsal="01")

2026-03-15 19:56:58,486 - ERROR - __main__ - Dorsal inválido '01': cero a la izquierda no permitido.


DorsalInvalidoError: Puedes usar '0' o '00', pero no se permiten ceros a la izquierda.

In [304]:
#EJEMPLO 4b: DORSAL INCORRECTO
player02 = Player(name="Markus Howard", dorsal="111")

2026-03-15 19:57:02,418 - ERROR - __main__ - Dorsal inválido '111': tiene más de dos cifras.


DorsalInvalidoError: El dorsal debe tener máximo dos cifras.

### **CLASE 'PlayerGameStats'**

Recoge las estadísticas (boxscore) de un jugador en un partido.


In [305]:
@dataclass
class PlayerGameStats:
    """
    Representa las estadísticas de un jugador en un partido concreto.
    Modelo: Boxscore ACB -> Reutilizamos código del TFA anterior.
    Usamos las mismas variables que un boxscore ACB, en el mismo orden.

    Añadimos validaciones de coherencia estadística. Respecto a la entrega anterior, se añaden mejoras. Ahora sí introducimos:
    1. PUNTOS. Comprobamos que puntos son coherentes con tiros anotados.
    2. VALORACIÓN. Comprobamos que la valoración introducida sea coherente con el resto de estadísticas.

    Por último, se incluyen métodos para calcular métricas avanzadas automáticamente.
    Diferenciamos dos tipos de métodos:

    A. Métodos con @property (PROPIEDADES) -> Dependen solo del jugador.
    podemos llamarlo como "stat.propiedad" en lugar de "stat.propiedad()". Así aligeramos la notación.

    B. Métodos convencionales, sin @property -> Dependen de parámetros externos
    Hay que llamarlos con paréntesis, "stat.método()".
    """

    player: "Player"

    minutos: int
    segundos: int

    puntos: int

    tl_anot: int
    tl_int: int

    t2_anot: int
    t2_int: int

    t3_anot: int
    t3_int: int

    reb_o: int
    reb_d: int

    ast: int
    perd: int
    rec: int

    tap_f: int
    tap_c: int

    mates: int

    fc: int
    fr: int

    plus_minus: int
    val: int

    def __post_init__(self):
        self._validate_stats()
        logger.info(f"'PlayerGameStats': Stats cargadas correctamente para {self.player.name} (#{self.player.dorsal})")

    # ==========================================================
    # VALIDACIONES
    # ==========================================================

    def _validate_stats(self):
        """
        Valida coherencia de las estadísticas del jugador.
        """

        # ----------------------------------------------------------------
        # 1. VALIDACIÓN GENERAL DATOS -> VALORES ENTEROS Y NO NEGATIVOS
        # ----------------------------------------------------------------

        integer_stats = [self.plus_minus] # El +- es la única estadística que puede ser negativa. -> Solo comprobamos que sea entero.
        non_negative_stats = [self.minutos, self.segundos,self.puntos,self.tl_anot, self.tl_int,self.t2_anot, self.t2_int,self.t3_anot, self.t3_int,self.reb_o, self.reb_d,self.ast, self.perd, self.rec,self.tap_f, self.tap_c,self.mates,self.fc, self.fr, self.val]


        for stat in (non_negative_stats + integer_stats):
            if not isinstance(stat, int):
              logger.error("Valor no entero detectado en estadísticas.")
              raise ValueError(f"Las estadísticas deben ser números enteros. Revisa el '{stat}'.")

        for stat in non_negative_stats:
            if stat < 0:
              logger.error("Valor negativo detectado en estadísticas.")
              raise ValueError(f"Las estadísticas no pueden ser negativas. Revisa el '{stat}'.")


        # --------------------------------------------------
        # 2. SEGUNDOS JUGADOS COHERENTES (Entre 0 y 59)
        # --------------------------------------------------

        if self.segundos >= 60:
            logger.error("Se detectó un dato de segundos incorrecto (>=60).")
            raise ValueError("Los segundos jugados deben estar entre 0 y 59.")

        # --------------------------------------------------
        # 3. TIROS COHERENTES
        # --------------------------------------------------

        if (self.tl_anot > self.tl_int):
            logger.error("Se detectó un error en los tiros libres.")
            raise InvalidStatError("TL anotados no pueden superar TL intentados.")

        if (self.t2_anot > self.t2_int):
            logger.error("Se detectó un error en los tiros de dos puntos.")
            raise InvalidStatError("T2 anotados no pueden superar T2 intentados.")

        if (self.t3_anot > self.t3_int):
            logger.error("Se detectó un error en los tiros de tres puntos.")
            raise InvalidStatError("T3 anotados no pueden superar T3 intentados.")

        # --------------------------------------------------
        # 4. PUNTOS COHERENTES CON TIROS ANOTADOS
        # --------------------------------------------------

        puntos_calculados = (self.tl_anot + 2 * self.t2_anot + 3 * self.t3_anot)

        if self.puntos != puntos_calculados:
            logger.error("Los puntos no coinciden con los tiros anotados.")
            raise InvalidStatError(f"Puntos incorrectos para el jugador {self.player.name}. "
                                   f"\nPuntos introducidos: {self.puntos} "
                                   f"\nDe acuerdo con los tiros anotados, los puntos anotados deberían ser: {puntos_calculados}.")

        # --------------------------------------------------
        # 5. FALTAS PERSONALES
        # --------------------------------------------------

        if self.fc > 5:
            logger.error("Se detectó un error en las faltas cometidas.")
            raise InvalidStatError("Un jugador no puede cometer más de 5 faltas.")

        # ----------------------------------------------------------------
        # 6. TAPONES RECIBIDOS COHERENTES CON TIROS DE CAMPO INTENTADOS.
        # ----------------------------------------------------------------

        tiros_campo_int = self.t2_int + self.t3_int

        if self.tap_c > tiros_campo_int:
            logger.error("Se detectó una inconsistencia entre tiros intentados y tapones recibidos.")
            raise InvalidStatError("No se pueden recibir más tapones que tiros de campo intentados.")

        # --------------------------------------------------
        # 7. MINUTOS JUGADOS REALISTAS
        # --------------------------------------------------

        if self.minutos > 90:
            logger.error("Se detectó un error en los minutos jugados.")
            raise InvalidStatError("Minutos jugados no realistas para un partido ACB.")

        # --------------------------------------------------
        # 8. VALORACIÓN ACB COHERENTE CON RESTO DE STATS
        # --------------------------------------------------

        valoracion_calculada = valoracion_acb(self.puntos, self.reb_o, self.reb_d, self.ast, self.rec, self.tap_f, self.fr, self.tl_anot, self.tl_int, self.t2_anot, self.t2_int, self.t3_anot, self.t3_int, self.perd, self.tap_c, self.fc)

        if self.val != valoracion_calculada:
            logger.error("Valoración ACB incorrecta.")
            raise InvalidStatError(f"Valoración incorrecta para el jugador {self.player.name}. "
                                   f"\nValoración introducida: {self.val} "
                                   f"\nDe acuerdo con las estadísticas introducidas, la valoración debería ser: {valoracion_calculada}.")


    # ==========================================================
    # MIS MÉTODOS -> ESTADÍSTICA CONVENCIONAL (Propiedades)
    # ==========================================================

    # IMPORTANTE -> QUE EL ORDEN LAS VARIABLES EN EL MÉTODO SEA EL MISMO QUE PARA LAS FUNCIONES
    # Recordamos que las variables en las funciones son mudas -> No importa el nombre, IMPORTA LA POSICIÓN

    @property
    def min_played(self):
        return min_played(self.minutos, self.segundos)

    @property
    def tl_pct(self):
        return tl_pct(self.tl_anot, self.tl_int)

    @property
    def t2_pct(self):
        return t2_pct(self.t2_anot, self.t2_int)

    @property
    def t3_pct(self):
        return t3_pct(self.t3_anot, self.t3_int)

    @property
    def fg_anot(self):
        return fg_anot(self.t2_anot, self.t3_anot)

    @property
    def fg_int(self):
        return fg_int(self.t2_int, self.t3_int)

    @property
    def fg_pct(self):
        return fg_pct(self.fg_anot, self.fg_int)


    @property
    def reb_tot(self):
        return reb_tot(self.reb_o, self.reb_d)

    @property
    def missed_shots(self):
        return missed_shots(self.tl_anot, self.tl_int, self.t2_anot, self.t2_int, self.t3_anot, self.t3_int)

    @property
    def valoracion_acb(self):
        return valoracion_acb(self.puntos,self.reb_o, self.reb_d,self.ast,self.rec,self.tap_f,self.fr,self.tl_anot, self.tl_int,self.t2_anot, self.t2_int,self.t3_anot, self.t3_int,self.perd,self.tap_c,self.fc)

    # ==========================================================
    # MIS MÉTODOS -> ESTADÍSTICA AVANZADA (Propiedades)
    # ==========================================================

    @property
    def posesiones_consumidas(self):
        return posesiones_consumidas(self.tl_int,self.t2_int,self.t3_int,self.perd)

    @property
    def oer_player(self):
        return oer_player(self.puntos,self.posesiones_consumidas)

    @property
    def efg(self):
        return efg(self.t2_anot,self.t2_int,self.t3_anot,self.t3_int)

    @property
    def ts(self):
        return ts(self.puntos,self.tl_int,self.t2_int,self.t3_int)

    @property
    def ft_rate(self):
        return ft_rate(self.tl_int, self.t2_int, self.t3_int)

    @property
    def ast_tov_ratio(self):
        return ast_tov_ratio(self.ast,self.perd)

    @property
    def ast_ratio_player(self):
        return ast_ratio_player(self.ast, self.tl_int, self.t2_int, self.t3_int, self.perd)

    @property
    def tov_ratio_player(self):
        return tov_ratio_player(self.tl_int, self.t2_int, self.t3_int, self.perd, self.ast)

    # ==========================================================================
    # MIS MÉTODOS -> ESTADÍSTICA AVANZADA: Métodos convencionales
    # USG% , USG%_TOT & AST%

    # Necesitan datos del equipo para calcularse: Minutos y posesiones totales.
    # Por eso no se puede usar @property.
    # ==========================================================================

    # USG% (Uso durante tiempo en pista)
    def usg(self, team_min, team_poss):
        return usg(self.tl_int,self.t2_int,self.t3_int,self.perd,self.min_played,team_min,team_poss)

    # USG%_TOTAL (Uso respecto al total del equipo en el partido)
    def usg_tot(self, team_poss):
        return usg_tot(self.tl_int,self.t2_int,self.t3_int,self.perd,team_poss)

    # AST%
    def ast_pct_player(self,t2_int_team,t3_int_team,min_team):
        return ast_pct_player(self.ast,self.t2_int,self.t3_int,self.min_played,t2_int_team,t3_int_team,min_team)


    # ==========================================================
    # MIS MÉTODOS -> IMPRIMIR RESUMEN DE STATS DE UN JUGADOR
    # ==========================================================

    @property
    def summary(self):
        """Imprime un resumen de las estadísticas del jugador."""

        print(f"\n================== RESUMEN DE ESTADÍSTICAS (Convencionales & Avanzadas) ==================\n")
        print(f"Jugador: {self.player.name}")
        print(f"Dorsal: #{self.player.dorsal}")


        print("\n---------- TIEMPO DE JUEGO ----------")
        print(f"Minutos jugados: {self.min_played:.2f}")
        print("MM:SS", format_minutes(self.min_played))

        print("\n---------- ANOTACIÓN ----------")
        print(f"Puntos: {self.puntos}")
        print(f"TL: {self.tl_anot}/{self.tl_int} ({round(self.tl_pct,2)}%)")
        print(f"T2: {self.t2_anot}/{self.t2_int} ({round(self.t2_pct,2)}%)")
        print(f"T3: {self.t3_anot}/{self.t3_int} ({round(self.t3_pct,2)}%)")
        print(f"FG: {self.fg_anot}/{self.fg_int} ({round(self.fg_pct,2)}%)")

        print("\n---------- REBOTES ----------")
        print(f"Ofensivos: {self.reb_o}")
        print(f"Defensivos: {self.reb_d}")
        print(f"Totales: {self.reb_tot}")

        print("\n---------- JUEGO ----------")
        print(f"Asistencias: {self.ast}")
        print(f"Pérdidas: {self.perd}")
        print(f"Recuperaciones: {self.rec}")

        print("\n---------- DEFENSA ----------")
        print(f"Tapones a favor: {self.tap_f}")
        print(f"Tapones recibidos: {self.tap_c}")

        print("\n---------- OTROS ----------")
        print(f"Mates: {self.mates}")
        print(f"Faltas cometidas: {self.fc}")
        print(f"Faltas recibidas: {self.fr}")

        print("\n---------- IMPACTO ----------")
        print(f"+/-: {self.plus_minus}")
        print(f"Valoración ACB: {self.val}")

        print("\n---------- MÉTRICAS AVANZADAS ----------")
        print(f"Posesiones consumidas: {round(self.posesiones_consumidas,2)}")
        print(f"OER: {round(self.oer_player,2)}\n")
        print(f"eFG%: {round(self.efg,2)}%")
        print(f"TS%: {round(self.ts,2)}%")
        print(f"FT_RATE: {round(self.ft_rate,2)}\n")
        print(f"AST/TOV: {round(self.ast_tov_ratio,2)}")
        print(f"AST_RATIO%: {round(self.ast_ratio_player,2)}%")
        print(f"AST%: {round(self.ast_pct_player(t2_int_team=48,t3_int_team=23,min_team=200),2)}%")
        print(f"TOV_RATIO: {round(self.tov_ratio_player,2)}%")

        print("\n---------- USAGE ----------")
        print(f"USG%: {round(self.usg(team_min=200, team_poss=92),2)}%")
        print(f"USG_total%: {round(self.usg_tot(team_poss=92),2)}%")

        print(f"\n====================================")


    # ==========================================================
    # MIS MÉTODOS -> CONVERTIR STATS EN DICCIONARIO
    # ==========================================================

    @property
    def to_dict(self):
        """Convierte las estadísticas (boxscore) del jugador a diccionario."""

        return {
            "jugador": self.player.name,
            "dorsal": self.player.dorsal,

            "min_played": self.min_played,
            "puntos": self.puntos,

            "tl_anot": self.tl_anot,
            "tl_int": self.tl_int,
            "tl_pct": self.tl_pct,

            "t2_anot": self.t2_anot,
            "t2_int": self.t2_int,
            "t2_pct": self.t2_pct,

            "t3_anot": self.t3_anot,
            "t3_int": self.t3_int,
            "t3_pct": self.t3_pct,

            #"fg_anot": self.fg_anot,
            #"fg_int": self.fg_int,
            #"fg_pct": self.fg_pct,

            "reb_o": self.reb_o,
            "reb_d": self.reb_d,
            "reb_tot": self.reb_tot,

            "ast": self.ast,
            "perd": self.perd,
            "rec": self.rec,

            "tap_f": self.tap_f,
            "tap_c": self.tap_c,

            "mates": self.mates,

            "fc": self.fc,
            "fr": self.fr,

            "plus_minus": self.plus_minus,
            "val": self.val

            #"efg": self.efg,
            #"ts": self.ts,
            #"oer": self.oer,
            #"ast_to": self.ast_to
        }

    # ==========================================================
    # MIS MÉTODOS -> CONVERTIR STATS EN DATAFRAME (PANDAS)
    # ==========================================================

    '''def to_dataframe(self):
        """Convierte las estadísticas del jugador a DataFrame."""
        return pd.DataFrame([self.to_dict()]) '''


    # En lugar de como método "to_dataframe", lo defino como una función global.
    # Así evito repetir código en el resto de objetos ->

    # Véase la funcion "to_dataframe" arriba, al inicio del código.




---
---

In [306]:
# EJEMPLO 1a -> EJEMPLO INCORRECTO de 'PlayerGameStats'
# COMPRUEBO VALIDACIÓN DE PUNTOS: Hacemos que los puntos no sean coherentes con los tiros anotados

# 1. Creamos el jugador
player_shai = Player("Shai Gilgeous-Alexander", "2")
# 2. Creamos sus estadísticas en un partido. Clase 'PlayerGameStats'
stats_shai = PlayerGameStats(player=player_shai,minutos=38,segundos=20,puntos=17,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=36)


2026-03-15 19:57:14,437 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:57:14,439 - ERROR - __main__ - Los puntos no coinciden con los tiros anotados.


NameError: name 'InvalidStatError' is not defined

In [308]:
# EJEMPLO 1b -> EJEMPLO INCORRECTO de 'PlayerGameStats'
# COMPRUEBO VALIDACIÓN DE VALORACIÓN ACB: Hacemos que la valoración no sea coherente con el resto de estadísticas.

# 1. Creamos el jugador
player_shai = Player("Shai Gilgeous-Alexander", "2")
# 2. Creamos sus estadísticas en un partido. Clase 'PlayerGameStats'
stats_shai = PlayerGameStats(player=player_shai,minutos=38,segundos=20,puntos=31,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=7)


2026-03-15 19:57:20,220 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:57:20,224 - ERROR - __main__ - Valoración ACB incorrecta.


NameError: name 'InvalidStatError' is not defined

In [307]:
# EJEMPLO 2 -> EJEMPLO CORRECTO de 'PlayerGameStats'

# 1. Creamos el jugador
player_shai = Player("Shai Gilgeous-Alexander", "2")

# 2. Creamos sus estadísticas en un partido. Clase 'PlayerGameStats'
stats_shai = PlayerGameStats(player=player_shai,minutos=38,segundos=20,puntos=31,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=36)

# 3. Probamos a calcular algunas estadísticas convencionales
print("\n--------------------------\n")
print("ESTADÍSTICA CONVENCIONAL:\n")

print("Minutos jugados (decimal):", round(stats_shai.min_played,2))
print("MM:SS", format_minutes(stats_shai.min_played))
print(f"\nFT%: {round(stats_shai.tl_pct,2)}%")
print(f"FG%: {round(stats_shai.fg_pct,2)}%")
print(f"3P%: {round(stats_shai.t3_pct,2)}%\n")
print("Rebotes totales:", stats_shai.reb_tot)
print("Valoración ACB:", stats_shai.valoracion_acb)

# 4. Probamos a calcular algunas estadísticas avanzadas
print("\n--------------------------\n")
print("ESTADÍSTICA AVANZADA: \n")

print("Posesiones consumidas:", stats_shai.posesiones_consumidas)
print(f"eFG%: {round(stats_shai.efg,2)}%")
print("AST/TOV:", round(stats_shai.ast_tov_ratio,2))

# 5. Probamos a calcular los "usage", que dependen de estadísticas de equipo
# Suponemos que el equipo jugó un total de 200 minutos, y tuvo 92 posesiones en el partido.
print("\n--------------------------\n")
print("USAGE: \n")

print(f"USG%: {round(stats_shai.usg(team_min=240, team_poss=92),2)}%")
print(f"USG_total%: {round(stats_shai.usg_tot(team_poss=92),2)}%")


2026-03-15 19:57:17,337 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:57:17,339 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)



--------------------------

ESTADÍSTICA CONVENCIONAL:

Minutos jugados (decimal): 38.33
MM:SS 38:20

FT%: 90.0%
FG%: 50.0%
3P%: 40.0%

Rebotes totales: 5
Valoración ACB: 36

--------------------------

ESTADÍSTICA AVANZADA: 

Posesiones consumidas: 27.4
eFG%: 55.0%
AST/TOV: 2.33

--------------------------

USAGE: 

USG%: 37.29%
USG_total%: 29.78%


In [309]:
# EJEMPLO 2 -> EJEMPLO CORRECTO de 'PlayerGameStats'

# 6. Veamos un resumen de todas las estadísticas del jugador:

stats_shai.summary


================== RESUMEN DE ESTADÍSTICAS (Convencionales & Avanzadas) ==================

Jugador: Shai Gilgeous-Alexander
Dorsal: #2

---------- TIEMPO DE JUEGO ----------
Minutos jugados: 38.33
MM:SS 38:20

---------- ANOTACIÓN ----------
Puntos: 31
TL: 9/10 (90.0%)
T2: 8/15 (53.33%)
T3: 2/5 (40.0%)
FG: 10/20 (50.0%)

---------- REBOTES ----------
Ofensivos: 1
Defensivos: 4
Totales: 5

---------- JUEGO ----------
Asistencias: 7
Pérdidas: 3
Recuperaciones: 2

---------- DEFENSA ----------
Tapones a favor: 0
Tapones recibidos: 1

---------- OTROS ----------
Mates: 1
Faltas cometidas: 2
Faltas recibidas: 8

---------- IMPACTO ----------
+/-: 12
Valoración ACB: 36

---------- MÉTRICAS AVANZADAS ----------
Posesiones consumidas: 27.4
OER: 1.13

eFG%: 55.0%
TS%: 63.52%


TypeError: ft_rate() takes 1 positional argument but 3 were given

In [310]:
# EJEMPLO 2 -> EJEMPLO CORRECTO de 'PlayerGameStats'

# 7. Finalmente, tenemos las estadísticas en formato Dataframe

df = to_dataframe(stats_shai.to_dict)
df

,jugador,dorsal,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,t2_pct,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
0,Shai Gilgeous-Alexander,2,38.33,31,9,10,90.0,8,15,53.33,...,7,3,2,0,1,1,2,8,12,36


### **CLASE 'TeamGameStats'**

Recoge las estadísticas asociadas el EQUIPO en un partido.

Esto es, aquellas que estdísticas que se contabilizan, pero NO se asignan a un jugador concreto: Rebotes de equipo, robos, pérdidas y faltas técnicas al banquillo o entrenador.

Corresponde con la la penúltima fila del boxscore ACB, siendo la última la fila de TOTALES.

In [311]:
@dataclass
class TeamGameStats:
    """
    Recoge las estadísticas del equipo que NO se asignan a jugadores concretos, es decir:

    - Rebotes de equipo
    - Pérdidas/recuperaciones de equipo
    - Faltas técnicas al banquillo o entrenador
    """

    reb_o: int
    reb_d: int

    perd: int
    rec: int

    fc: int

    def __post_init__(self):
        self._validate_stats()
        logger.info(f"'TeamGameStats': Stats de equipo cargadas correctamente.")

    # ==========================================================
    # VALIDACIONES
    # ==========================================================

    def _validate_stats(self):
        """
        Valida coherencia de las estadísticas del equipo.
        """

        non_negative_stats = [self.reb_o, self.reb_d, self.perd, self.rec, self.fc]

        # ----------------------------------------------------------------
        # 1. VALIDACIÓN GENERAL DATOS -> VALORES ENTEROS Y NO NEGATIVOS
        # ----------------------------------------------------------------

        for stat in non_negative_stats:

            if not isinstance(stat, int):
              logger.error("Valor no entero detectado en estadísticas de equipo.")
              raise ValueError("Las estadísticas de equipo deben ser números enteros.")

            if stat < 0:
              logger.error("Valor negativo detectado en estadísticas de equipo.")
              raise ValueError("Las estadísticas de equipo no pueden ser negativas.")

        #---------------------------------------------------------------------------------------
        # VALIDACIÓN: tipo entero
        if any(not isinstance(stat, int) for stat in non_negative_stats):
            logger.error("Valor no entero detectado en estadísticas de equipo.")
            raise ValueError(f"Las estadísticas deben ser números enteros. Revisa el '{stat}'.")


        # VALIDACIÓN: no negativos
        if any(stat < 0 for stat in non_negative_stats):
            logger.error("Valor negativo detectado en estadísticas de equipo.")
            raise ValueError(f"Las estadísticas no pueden ser negativas. Revisa el '{stat}'.")

        self.val = self.reb_o+self.reb_d+self.rec-self.perd-self.fc


    # ==========================================================
    # MÉTODOS
    # ==========================================================

    @property
    def reb_tot(self):
        """Rebotes totales del equipo"""
        return self.reb_o + self.reb_d

    @property
    def to_dict(self):
        """
        Convierte las estadísticas del equipo a diccionario, con el mismo formato que PlayerGameStats.
        Esto permite integrarlo fácilmente en DataFrames.
        """

        return {
            "jugador": "Equipo",
            "dorsal": "",

            "min_played": 0,

            "puntos": 0,

            "tl_anot": 0,
            "tl_int": 0,
            "tl_pct": 0,

            "t2_anot": 0,
            "t2_int": 0,
            "t2_pct": 0,


            "t3_anot": 0,
            "t3_int": 0,
            "t3_pct": 0,

            "reb_o": self.reb_o,
            "reb_d": self.reb_d,
            "reb_tot": self.reb_tot,

            "ast": 0,
            "perd": self.perd,
            "rec": self.rec,

            "tap_f": 0,
            "tap_c": 0,

            "mates": 0,

            "fc": self.fc,
            "fr": 0,

            "plus_minus": 0,
            "val": self.val,
        }

In [312]:
# EJEMPLO INCORRECTO de "TeamGameStats"
team_stats = TeamGameStats(reb_o=2.3,reb_d=1,perd=1,rec=0,fc=1)


2026-03-15 19:57:34,639 - ERROR - __main__ - Valor no entero detectado en estadísticas de equipo.


ValueError: Las estadísticas de equipo deben ser números enteros.

In [313]:
# EJEMPLO CORRECTO de "TeamGameStats"
team_stats = TeamGameStats(reb_o=2,reb_d=1,perd=1,rec=0,fc=1)
df = to_dataframe(team_stats.to_dict)
df

2026-03-15 19:57:36,780 - INFO - __main__ - 'TeamGameStats': Stats de equipo cargadas correctamente.


,jugador,dorsal,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,t2_pct,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
0,Equipo,,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,1


### **CLASE 'MyTeamGame'**



In [314]:
@dataclass
class MyTeamGame:
    """
    Contiene el boxscore completo de mi equipo en un partido. Es decir, incluye:

    - Estadísticas individuales de jugadores
    - Estadísticas de equipo (TEAM)

    - Permite calcular ESTADÍSTICAS TOTALES (Jugadores + Equipo)
    """

    team_name: str
    players_list: List["Player"]
    stats_list: List["PlayerGameStats"]
    team_stats: "TeamGameStats" = None

    _used_dorsals: set = field(default_factory=set, init=False, repr=False)

    def __post_init__(self):
        self._validate_info()
        logger.info(f"'MyTeamGame': Stats cargadas correctamente para '{self.team_name}' con {len(self.players_list)} jugadores")


    # ==========================================================
    # VALIDACIONES
    # ==========================================================

    def _validate_info(self):
        """
        Valida la coherencia de la información completa del partido de mi equipo:
        Lista de jugadores, lista de estadísticas, nº de jugadores en acta, coherencia de dorsales utilizados...
        """

        # ----------------------------------------------------------------------------------
        # 1. AMBAS LISTAS CON MISMA LONGITUD (Lista jugadores / Lista de sus estadísticas)
        # ----------------------------------------------------------------------------------

        if len(self.players_list) != len(self.stats_list):
            logger.error("Número de jugadores y número de líneas estadísticas no coincide")
            raise ValueError("Número de jugadores y número de líneas estadísticas no coincide")

        # -----------------------------------------------------------------------
        # 2. NÚMERO DE JUGADORES (5-12)
        #    Debe haber entre 5 y 12 jugadores inscritos en el acta del partido.
        # -----------------------------------------------------------------------

        if len(self.players_list) < 5 or len(self.players_list) > 12:
            logger.error(f"Intento de crear equipo con {len(self.players_list)} jugadores. Deben añadirse entre 5 y 12 jugadores")
            raise NumeroJugadoresError("El equipo debe tener entre 5 y 12 jugadores.")

        # --------------------------------------------------
        # 3. COHERENCIA PLAYER / STATS
        #    Debemos comprobar que cada estadística pertenece al jugador correcto
        #    y que no hay dorsales duplicados.
        # --------------------------------------------------

        for player, stats in zip(self.players_list, self.stats_list):

            if stats.player is not player:
                logger.error(f"Parece que las estadísticas de {player.name} no se han añadido correctamente")
                raise PlayerYStatsNoCorrespondientesError(f"Hay un problema. Revisa la lista de tus jugadores y la lista de sus estadísticas. \nComprueba que ambas listas coinciden en el orden adecuado. \nParece que las estadísticas de {player.name} no se han añadido correctamente.")

            dorsal = player.dorsal

            if dorsal in self._used_dorsals:
                logger.error(f"Dorsal duplicado al crear partido: {dorsal}")
                raise DorsalDuplicadoEnPartidoError(f"El dorsal {dorsal} ya está en uso en este partido.")

            self._used_dorsals.add(dorsal)

        logger.info(f"'MyTeamGame' creado para '{self.team_name}' con {len(self.players_list)} jugadores")



    # ==========================================================
    # MÉTODO -> CALCULAR ESTADÍSTICA ACUMULADA DEL PARTIDO:
    # TOTALES = Jugadores + Equipo.
    # ==========================================================

    @property
    def team_totals(self):
        """
        Calcula los totales del equipo sumando:
        - estadísticas de todos los jugadores
        - estadísticas de TEAM (si existen)

        Además, devuelve estas estadísticas acumuladas en formato DICCIONARIO, para integrarlo en DataFrames
        Mismo formato que en PlayerGameStats y TeamGameStats.
        """

        totals = {
            "jugador": f"Total ({self.team_name})",
            "dorsal": "",
            "min_played": sum(s.min_played for s in self.stats_list),
            "puntos": sum(s.puntos for s in self.stats_list),
            "tl_anot": sum(s.tl_anot for s in self.stats_list),
            "tl_int": sum(s.tl_int for s in self.stats_list),
            "tl_pct": 0, #Inicializamos para que guarde la posición dentro del diccionario. Lo calculamos después
            "t2_anot": sum(s.t2_anot for s in self.stats_list),
            "t2_int": sum(s.t2_int for s in self.stats_list),
            "t2_pct": 0,
            "t3_anot": sum(s.t3_anot for s in self.stats_list),
            "t3_int": sum(s.t3_int for s in self.stats_list),
            "t3_pct": 0,
            "reb_o": sum(s.reb_o for s in self.stats_list),
            "reb_d": sum(s.reb_d for s in self.stats_list),
            "reb_tot": sum(s.reb_tot for s in self.stats_list),
            "ast": sum(s.ast for s in self.stats_list),
            "perd": sum(s.perd for s in self.stats_list),
            "rec": sum(s.rec for s in self.stats_list),
            "tap_f": sum(s.tap_f for s in self.stats_list),
            "tap_c": sum(s.tap_c for s in self.stats_list),
            "mates": sum(s.mates for s in self.stats_list),
            "fc": sum(s.fc for s in self.stats_list),
            "fr": sum(s.fr for s in self.stats_list),
            "plus_minus": sum(s.plus_minus for s in self.stats_list),
            "val": sum(s.val for s in self.stats_list),
        }

        # Añadimos porcentajes de tiro
        totals["tl_pct"] = tl_pct(totals["tl_anot"], totals["tl_int"])
        totals["t2_pct"] = t2_pct(totals["t2_anot"], totals["t2_int"])
        totals["t3_pct"] = t3_pct(totals["t3_anot"], totals["t3_int"])

        # Añadimos estadísticas del Equipo, si existen
        if self.team_stats:

            totals["reb_o"] += self.team_stats.reb_o
            totals["reb_d"] += self.team_stats.reb_d
            totals["reb_tot"] += self.team_stats.reb_tot
            totals["perd"] += self.team_stats.perd
            totals["rec"] += self.team_stats.rec
            totals["fc"] += self.team_stats.fc
            totals["val"] += self.team_stats.val

        return totals


    # ============================================================================
    # MÉTODOS PRINCIPALES -> MOSTRAR BOXSCORE COMPLETO DE MI EQUIPO EN EL PARTIDO
    # Implementamos dos métodos diferentes
    # ============================================================================

    # MÉTODO I. PRINCIPAL -> BOXSCORE INTERNO. Para cálculos y análisis. No modifica datos: nombres de columnas, etc.
    @property
    def boxscore(self):

        rows = []

        # JUGADORES
        for stats in self.stats_list:
            rows.append(stats.to_dict)

        # EQUIPO
        if self.team_stats:
            rows.append(self.team_stats.to_dict)

        # TOTALES
        rows.append(self.team_totals)

        # CREAR DATA FRAME
        df = pd.DataFrame(rows)

        # ----------------------------------------------------------
        # ORDENAR POR DORSAL (00 -> 0 -> 1 -> 2 ...)
        # ----------------------------------------------------------

        df["_dorsal_sort"] = df["dorsal"].apply(lambda x: (int(x) if str(x).isdigit() else 999, str(x)))
        df = df.sort_values("_dorsal_sort")
        df.drop(columns="_dorsal_sort", inplace=True)
        #df.set_index("dorsal", inplace=True)

        # ----------------------------------------------------------
        # AÑADIR ESPACIO ANTES DE LOS TOTALES
        # ----------------------------------------------------------

        separator = pd.DataFrame([{col: "" for col in df.columns}])

        df = pd.concat([df.iloc[:-1],separator,df.iloc[-1:]],ignore_index=True)

        return df


    # MÉTODO II. ALTERNATIVO: BOXSCORE EN FORMATO ACB. Visualmente más bonito.
    @property
    def boxscore_acb(self):
        '''
        BOXSCORE FORMATO ACB -> Cambia nombres y orden de las columnas.
        Coincide casi exactamente con el formato del Boxscore de ACB,  salvo algunos nombres de columnas, que me resultan confusos.
        Ejemplo: Prefiero "FC" y "FR" para "Faltas Cometidas/Recibidas" que "PF" y "FD".
        '''

        # COPIA DEL DATAFRAME INTERNO
        display_df = self.boxscore.copy()

        # Renombrar columnas -> FORMATO BOXSCORE ACB
        column_names = {
            "jugador": "PLAYER",
            "dorsal": "DORSAL",
            "puntos": "PTS",
            "min_played": "MIN",
            "tl_anot": "TL_Anot",
            "tl_int": "TL_Int",
            "tl_pct": "TL%",
            "t2_anot": "T2_Anot",
            "t2_int": "T2_Int",
            "t2_pct": "T2%",
            "t3_anot": "T3_Anot",
            "t3_int": "T3_Int",
            "t3_pct": "T3%",
            "reb_d": "DR",
            "reb_o": "OR",
            "reb_tot": "TR",
            "ast": "AST",
            "perd": "TO",
            "rec": "STL",
            "tap_f": "BLK",
            "tap_c": "BLK_Rec",
            "mates": "DUNK",
            "fc": "FC",
            "fr": "FR",
            "plus_minus": "+/-",
            "val": "VAL"
        }

        display_df.rename(columns=column_names, inplace=True)

        ordered_cols = [
            "DORSAL",
            "PLAYER",
            "PTS",
            "MIN",
            "T2_Anot","T2_Int","T2%",
            "T3_Anot","T3_Int","T3%",
            "TL_Anot","TL_Int","TL%",
            "DR","OR","TR",
            "AST","TO","STL",
            "BLK","BLK_Rec",
            "DUNK",
            "FC","FR",
            "+/-","VAL"
        ]

        display_df = display_df[ordered_cols]

        # ----------------------------------------------------------
        # FORMATO MINUTOS (MM:SS)
        # ----------------------------------------------------------

        display_df["MIN"] = display_df["MIN"].apply(format_minutes)

        # ----------------------------------------------------------
        # MOSTRAR 2 DECIMALES SIN CAMBIAR DATOS
        # ----------------------------------------------------------

        display_df = display_df.round(2)
        return display_df


    # MÉTODO FINAL. CREAR DATAFRAME SOLO CON LOS JUGADORES ->
    # Para cálculos de estadística avanzada con métricas solo aplicables a jugadores

    @property
    def dataframe_only_players(self):
        rows = []

        # JUGADORES
        for stats in self.stats_list:
            rows.append(stats.to_dict)

        # CREAR DATA FRAME
        df = pd.DataFrame(rows)

        # ----------------------------------------------------------
        # ORDENAR POR DORSAL (00 -> 0 -> 1 -> 2 ...)
        # ----------------------------------------------------------

        df["_dorsal_sort"] = df["dorsal"].apply(lambda x: (int(x) if str(x).isdigit() else 999, str(x)))
        df = df.sort_values("_dorsal_sort")
        df.drop(columns="_dorsal_sort", inplace=True)

        return df

Antes de continuar con las siguientes clases, vamos a poner varios ejemplos de "MyTeamGame".

- EJEMPLO 1: SELECCIÓN ESPAÑOLA. Mostramos un ejemplo correcto, con la carga de los jugadores, equipo, y mostrando ambos tipos de boxscore

- EJEMPLO 2: OKC THUNDER. Mostramos varios ejemplos incorrectos, de fallos posibles al cargar los datos, uso de dorsales duplicados en un partido, etc. Finalmente, corregimos estos errores, y mostramos otro ejemplo correcto.

#### **CLASE 'MyTeamGame' - EJEMPLO 1: SELECCIÓN ESPAÑOLA**


In [315]:
# EJEMPLO 1: SELECCIÓN ESPAÑOLA - EJEMPLO CORRECTO de "MyTeamGame"

# ==========================================
# CREAR JUGADORES: 'Player'
# ==========================================

ricky = Player("Ricky Rubio", "9")
calderon = Player("Jose Manuel Calderon", "8")
chacho = Player("Sergio Rodriguez", "13")
llull = Player("Sergio Llull", "23")
navarro = Player("Juan Carlos Navarro", "7")
rudy = Player("Rudy Fernandez", "5")
ibaka = Player("Serge Ibaka", "14")
garbajosa = Player("Jorge Garbajosa", "15")
pau = Player("Pau Gasol", "4")
marc = Player("Marc Gasol", "33")

# ==========================================
# CREAR 'PlayerGameStats'
# Minutos totales = 200
# ==========================================

ricky_stats = PlayerGameStats(player=ricky,minutos=22,segundos=30,puntos=8,tl_anot=2,tl_int=2,t2_anot=3,t2_int=5,t3_anot=0,t3_int=2,reb_o=1,reb_d=3,ast=7,perd=2,rec=2,tap_f=0,tap_c=0,mates=0,fc=2,fr=3,plus_minus=5,val=16)
calderon_stats = PlayerGameStats(player=calderon,minutos=21,segundos=30,puntos=10,tl_anot=2,tl_int=2,t2_anot=1,t2_int=3,t3_anot=2,t3_int=4,reb_o=0,reb_d=2,ast=4,perd=1,rec=1,tap_f=0,tap_c=0,mates=0,fc=1,fr=2,plus_minus=6,val=13)
chacho_stats = PlayerGameStats(player=chacho,minutos=22,segundos=0,puntos=7,tl_anot=1,tl_int=2,t2_anot=3,t2_int=5,t3_anot=0,t3_int=2,reb_o=0,reb_d=2,ast=6,perd=2,rec=1,tap_f=0,tap_c=0,mates=0,fc=1,fr=1,plus_minus=4,val=9)
llull_stats = PlayerGameStats(player=llull,minutos=22,segundos=0,puntos=11,tl_anot=2,tl_int=2,t2_anot=3,t2_int=6,t3_anot=1,t3_int=3,reb_o=0,reb_d=2,ast=3,perd=1,rec=1,tap_f=0,tap_c=0,mates=0,fc=2,fr=3,plus_minus=8,val=12)
navarro_stats = PlayerGameStats(player=navarro,minutos=22,segundos=0,puntos=14,tl_anot=4,tl_int=4,t2_anot=2,t2_int=4,t3_anot=2,t3_int=5,reb_o=0,reb_d=1,ast=2,perd=1,rec=1,tap_f=0,tap_c=0,mates=0,fc=1,fr=4,plus_minus=7,val=15)
rudy_stats = PlayerGameStats(player=rudy,minutos=18,segundos=0,puntos=11,tl_anot=4,tl_int=4,t2_anot=2,t2_int=4,t3_anot=1,t3_int=3,reb_o=1,reb_d=3,ast=2,perd=1,rec=2,tap_f=1,tap_c=0,mates=1,fc=2,fr=3,plus_minus=6,val=16)
ibaka_stats = PlayerGameStats(player=ibaka,minutos=18,segundos=0,puntos=9,tl_anot=1,tl_int=2,t2_anot=4,t2_int=6,t3_anot=0,t3_int=0,reb_o=2,reb_d=4,ast=0,perd=1,rec=1,tap_f=3,tap_c=0,mates=2,fc=2,fr=2,plus_minus=5,val=15)
garbajosa_stats = PlayerGameStats(player=garbajosa,minutos=18,segundos=0,puntos=6,tl_anot=0,tl_int=0,t2_anot=3,t2_int=5,t3_anot=0,t3_int=2,reb_o=1,reb_d=3,ast=1,perd=1,rec=0,tap_f=0,tap_c=0,mates=0,fc=2,fr=1,plus_minus=3,val=5)
pau_stats = PlayerGameStats(player=pau,minutos=18,segundos=0,puntos=16,tl_anot=4,tl_int=5,t2_anot=6,t2_int=10,t3_anot=0,t3_int=0,reb_o=3,reb_d=6,ast=2,perd=2,rec=1,tap_f=2,tap_c=1,mates=1,fc=3,fr=4,plus_minus=9,val=23)
marc_stats = PlayerGameStats(player=marc,minutos=18,segundos=0,puntos=13,tl_anot=3,tl_int=4,t2_anot=5,t2_int=8,t3_anot=0,t3_int=1,reb_o=2,reb_d=5,ast=3,perd=1,rec=1,tap_f=1,tap_c=1,mates=1,fc=2,fr=3,plus_minus=7,val=19)

# ==========================================
# CREAR 'TeamGameStats'
# ==========================================

team_stats_esp = TeamGameStats(reb_o=1,reb_d=2,perd=1,rec=0,fc=1)

# ==========================================
# CREAR 'MyTeamGame'
# ==========================================

spain_game = MyTeamGame(
    team_name="España",
    players_list=[ricky, calderon, chacho, llull, navarro,rudy, ibaka, garbajosa, pau, marc],
    stats_list=[ricky_stats, calderon_stats, chacho_stats, llull_stats, navarro_stats,rudy_stats, ibaka_stats, garbajosa_stats, pau_stats, marc_stats],
    team_stats=team_stats_esp
)

# ==========================================
# VER TOTALES DEL EQUIPO
# ==========================================

spain_game.team_totals

2026-03-15 19:57:47,749 - INFO - __main__ - Jugador creado: Ricky Rubio (#9)
2026-03-15 19:57:47,751 - INFO - __main__ - Jugador creado: Jose Manuel Calderon (#8)
2026-03-15 19:57:47,753 - INFO - __main__ - Jugador creado: Sergio Rodriguez (#13)
2026-03-15 19:57:47,754 - INFO - __main__ - Jugador creado: Sergio Llull (#23)
2026-03-15 19:57:47,756 - INFO - __main__ - Jugador creado: Juan Carlos Navarro (#7)
2026-03-15 19:57:47,757 - INFO - __main__ - Jugador creado: Rudy Fernandez (#5)
2026-03-15 19:57:47,758 - INFO - __main__ - Jugador creado: Serge Ibaka (#14)
2026-03-15 19:57:47,759 - INFO - __main__ - Jugador creado: Jorge Garbajosa (#15)
2026-03-15 19:57:47,760 - INFO - __main__ - Jugador creado: Pau Gasol (#4)
2026-03-15 19:57:47,761 - INFO - __main__ - Jugador creado: Marc Gasol (#33)
2026-03-15 19:57:47,763 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Ricky Rubio (#9)
2026-03-15 19:57:47,764 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correc

{'jugador': 'Total (España)',
 'dorsal': '',
 'min_played': 200.0,
 'puntos': 105,
 'tl_anot': 23,
 'tl_int': 27,
 'tl_pct': 85.18518518518519,
 't2_anot': 32,
 't2_int': 56,
 't2_pct': 57.14285714285714,
 't3_anot': 6,
 't3_int': 22,
 't3_pct': 27.27272727272727,
 'reb_o': 11,
 'reb_d': 33,
 'reb_tot': 44,
 'ast': 30,
 'perd': 14,
 'rec': 11,
 'tap_f': 7,
 'tap_c': 2,
 'mates': 5,
 'fc': 19,
 'fr': 26,
 'plus_minus': 60,
 'val': 144}

In [316]:
# EJEMPLO 1: SELECCIÓN ESPAÑOLA - EJEMPLO CORRECTO de "MyTeamGame"

# ==========================================
# VER BOXSCORE (MODELO I. Interno)
# ==========================================
boxscore_spain = spain_game.boxscore
boxscore_spain

,jugador,dorsal,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,t2_pct,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
0,Pau Gasol,4,18.0,16,4,5,80.0,6,10,60.0,...,2,2,1,2,1,1,3,4,9,23
1,Rudy Fernandez,5,18.0,11,4,4,100.0,2,4,50.0,...,2,1,2,1,0,1,2,3,6,16
2,Juan Carlos Navarro,7,22.0,14,4,4,100.0,2,4,50.0,...,2,1,1,0,0,0,1,4,7,15
3,Jose Manuel Calderon,8,21.5,10,2,2,100.0,1,3,33.33,...,4,1,1,0,0,0,1,2,6,13
4,Ricky Rubio,9,22.5,8,2,2,100.0,3,5,60.0,...,7,2,2,0,0,0,2,3,5,16
5,Sergio Rodriguez,13,22.0,7,1,2,50.0,3,5,60.0,...,6,2,1,0,0,0,1,1,4,9
6,Serge Ibaka,14,18.0,9,1,2,50.0,4,6,66.67,...,0,1,1,3,0,2,2,2,5,15
7,Jorge Garbajosa,15,18.0,6,0,0,0.0,3,5,60.0,...,1,1,0,0,0,0,2,1,3,5
8,Sergio Llull,23,22.0,11,2,2,100.0,3,6,50.0,...,3,1,1,0,0,0,2,3,8,12
9,Marc Gasol,33,18.0,13,3,4,75.0,5,8,62.5,...,3,1,1,1,1,1,2,3,7,19


In [317]:
# EJEMPLO 1: SELECCIÓN ESPAÑOLA - EJEMPLO CORRECTO de "MyTeamGame"

# ==========================================
# VER BOXSCORE (MODELO II. Formato ACB)
# ==========================================

boxscore_spain_formato_acb = spain_game.boxscore_acb
boxscore_spain_formato_acb

,DORSAL,PLAYER,PTS,MIN,T2_Anot,T2_Int,T2%,T3_Anot,T3_Int,T3%,...,AST,TO,STL,BLK,BLK_Rec,DUNK,FC,FR,+/-,VAL
0,4,Pau Gasol,16,18:00,6,10,60.0,0,0,0.0,...,2,2,1,2,1,1,3,4,9,23
1,5,Rudy Fernandez,11,18:00,2,4,50.0,1,3,33.33,...,2,1,2,1,0,1,2,3,6,16
2,7,Juan Carlos Navarro,14,22:00,2,4,50.0,2,5,40.0,...,2,1,1,0,0,0,1,4,7,15
3,8,Jose Manuel Calderon,10,21:30,1,3,33.33,2,4,50.0,...,4,1,1,0,0,0,1,2,6,13
4,9,Ricky Rubio,8,22:30,3,5,60.0,0,2,0.0,...,7,2,2,0,0,0,2,3,5,16
5,13,Sergio Rodriguez,7,22:00,3,5,60.0,0,2,0.0,...,6,2,1,0,0,0,1,1,4,9
6,14,Serge Ibaka,9,18:00,4,6,66.67,0,0,0.0,...,0,1,1,3,0,2,2,2,5,15
7,15,Jorge Garbajosa,6,18:00,3,5,60.0,0,2,0.0,...,1,1,0,0,0,0,2,1,3,5
8,23,Sergio Llull,11,22:00,3,6,50.0,1,3,33.33,...,3,1,1,0,0,0,2,3,8,12
9,33,Marc Gasol,13,18:00,5,8,62.5,0,1,0.0,...,3,1,1,1,1,1,2,3,7,19


In [318]:
# EJEMPLO 1: SELECCIÓN ESPAÑOLA - EJEMPLO CORRECTO de "MyTeamGame"

# ==========================================
# VER DATAFRAME SOLO DE JUGADORES
# ==========================================

spain_only_players = spain_game.dataframe_only_players
spain_only_players

,jugador,dorsal,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,t2_pct,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
8,Pau Gasol,4,18.0,16,4,5,80.0,6,10,60.00,...,2,2,1,2,1,1,3,4,9,23
5,Rudy Fernandez,5,18.0,11,4,4,100.0,2,4,50.00,...,2,1,2,1,0,1,2,3,6,16
4,Juan Carlos Navarro,7,22.0,14,4,4,100.0,2,4,50.00,...,2,1,1,0,0,0,1,4,7,15
1,Jose Manuel Calderon,8,21.5,10,2,2,100.0,1,3,33.33,...,4,1,1,0,0,0,1,2,6,13
0,Ricky Rubio,9,22.5,8,2,2,100.0,3,5,60.00,...,7,2,2,0,0,0,2,3,5,16
2,Sergio Rodriguez,13,22.0,7,1,2,50.0,3,5,60.00,...,6,2,1,0,0,0,1,1,4,9
6,Serge Ibaka,14,18.0,9,1,2,50.0,4,6,66.67,...,0,1,1,3,0,2,2,2,5,15
7,Jorge Garbajosa,15,18.0,6,0,0,0.0,3,5,60.00,...,1,1,0,0,0,0,2,1,3,5
3,Sergio Llull,23,22.0,11,2,2,100.0,3,6,50.00,...,3,1,1,0,0,0,2,3,8,12
9,Marc Gasol,33,18.0,13,3,4,75.0,5,8,62.50,...,3,1,1,1,1,1,2,3,7,19


---

#### **CLASE 'MyTeamGame' - EJEMPLO 2: OKC THUNDER**


In [319]:
# EJEMPLO 2: OKC THUNDER
# EJEMPLO INCORRECTO 2-A: Stats que no pertenecen a un jugador
# CASO A  -> Nos damos cuenta del error al añadir las estadísticas del partido a los jugadores.

# ---------------------------------------------------
# 1. Creamos los jugadores
# ---------------------------------------------------

player_shai = Player("Shai Gilgeous-Alexander", "2")
player_dort = Player("Luguentz Dort", "5")
player_chet = Player("Chet Holmgren", "7")
player_jalen = Player("Jalen Williams", "8")
player_caruso = Player("Alex Caruso", "9")

# -----------------------------------------------------------------------------------------------
# 2. Creamos las estadísticas individuales de cada jugador en el partido. Clase 'PlayerGameStats'
# ERROR -> Asignamos las estadísticas "stats_dort" al jugador "player_shai"
# -----------------------------------------------------------------------------------------------

stats_shai = PlayerGameStats(player=player_shai,minutos=48,segundos=0,puntos=31,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=36)
stats_dort = PlayerGameStats(player=player_shai,minutos=48,segundos=0,puntos=14,tl_anot=2,tl_int=2,t2_anot=3,t2_int=6,t3_anot=2,t3_int=7,reb_o=1,reb_d=3,ast=2,perd=1,rec=1,tap_f=1,tap_c=0,mates=1,fc=3,fr=2,plus_minus=5,val=12)
stats_chet = PlayerGameStats(player=player_chet,minutos=48,segundos=0,puntos=19,tl_anot=4,tl_int=4,t2_anot=6,t2_int=10,t3_anot=1,t3_int=3,reb_o=3,reb_d=7,ast=2,perd=2,rec=1,tap_f=3,tap_c=1,mates=1,fc=2,fr=3,plus_minus=10,val=27)
stats_jalen = PlayerGameStats(player=player_jalen,minutos=48,segundos=0,puntos=22,tl_anot=5,tl_int=6,t2_anot=7,t2_int=12,t3_anot=1,t3_int=4,reb_o=2,reb_d=5,ast=4,perd=2,rec=2,tap_f=0,tap_c=0,mates=2,fc=1,fr=4,plus_minus=8,val=27)
stats_caruso = PlayerGameStats(player=player_caruso,minutos=48,segundos=0,puntos=10,tl_anot=1,tl_int=2,t2_anot=3,t2_int=5,t3_anot=1,t3_int=4,reb_o=1,reb_d=4,ast=5,perd=1,rec=3,tap_f=1,tap_c=0,mates=0,fc=2,fr=1,plus_minus=6,val=16)

# ---------------------------------------------------
# 3. Creamos las listas del partido
# LISTA I. Lista de jugadores en el acta del partido.
# LISTA II. Estadísticas de dichos jugadores.
# ---------------------------------------------------

j1_players = [player_shai, player_dort, player_chet, player_jalen, player_caruso]
j1_stats = [stats_shai, stats_dort, stats_chet, stats_jalen, stats_caruso]

# -------------------------------------------------------------------------------------------------------------------------
# 4. Añadimos a cada jugador sus stats en ese partido -> Estadísticas individuales de temporada.
# Lo hacemos con una función auxiliar, que usa zip() y comprueba que las stats correspondan efectivamente a cada jugador.
# -------------------------------------------------------------------------------------------------------------------------

add_game_to_players(j1_players, j1_stats)

2026-03-15 19:57:59,827 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:57:59,829 - INFO - __main__ - Jugador creado: Luguentz Dort (#5)
2026-03-15 19:57:59,831 - INFO - __main__ - Jugador creado: Chet Holmgren (#7)
2026-03-15 19:57:59,832 - INFO - __main__ - Jugador creado: Jalen Williams (#8)
2026-03-15 19:57:59,834 - INFO - __main__ - Jugador creado: Alex Caruso (#9)
2026-03-15 19:57:59,837 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)
2026-03-15 19:57:59,838 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)
2026-03-15 19:57:59,840 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Chet Holmgren (#7)
2026-03-15 19:57:59,841 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Jalen Williams (#8)
2026-03-15 19:57:59,842 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Alex Caruso (#

PlayerYStatsNoCorrespondientesError: Hay un problema. Revisa la lista de tus jugadores y la lista de sus estadísticas. 
Comprueba que ambas listas coinciden en el orden adecuado. 
Parece que las estadísticas de Luguentz Dort no se han añadido correctamente.

In [320]:
# EJEMPLO 2: OKC THUNDER
# EJEMPLO INCORRECTO 2-B: Stats que no pertenecen a un jugador
# CASO B -> Se nos olvida añadir las estadísticas del partido a los jugadores. Nos damos cuenta del error al crear 'MyTeamGame'

# ---------------------------------------------------
# 1. Creamos los jugadores
# ---------------------------------------------------

player_shai = Player("Shai Gilgeous-Alexander", "2")
player_dort = Player("Luguentz Dort", "5")
player_chet = Player("Chet Holmgren", "7")
player_jalen = Player("Jalen Williams", "8")
player_caruso = Player("Alex Caruso", "9")

# -----------------------------------------------------------------------------------------------
# 2. Creamos las estadísticas individuales de cada jugador en el partido. Clase 'PlayerGameStats'
# ERROR -> Asignamos las estadísticas "stats_dort" al jugador "player_shai"
# -----------------------------------------------------------------------------------------------

stats_shai = PlayerGameStats(player=player_shai,minutos=48,segundos=0,puntos=31,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=36)
stats_dort = PlayerGameStats(player=player_shai,minutos=48,segundos=0,puntos=14,tl_anot=2,tl_int=2,t2_anot=3,t2_int=6,t3_anot=2,t3_int=7,reb_o=1,reb_d=3,ast=2,perd=1,rec=1,tap_f=1,tap_c=0,mates=1,fc=3,fr=2,plus_minus=5,val=12)
stats_chet = PlayerGameStats(player=player_chet,minutos=48,segundos=0,puntos=19,tl_anot=4,tl_int=4,t2_anot=6,t2_int=10,t3_anot=1,t3_int=3,reb_o=3,reb_d=7,ast=2,perd=2,rec=1,tap_f=3,tap_c=1,mates=1,fc=2,fr=3,plus_minus=10,val=27)
stats_jalen = PlayerGameStats(player=player_jalen,minutos=48,segundos=0,puntos=22,tl_anot=5,tl_int=6,t2_anot=7,t2_int=12,t3_anot=1,t3_int=4,reb_o=2,reb_d=5,ast=4,perd=2,rec=2,tap_f=0,tap_c=0,mates=2,fc=1,fr=4,plus_minus=8,val=27)
stats_caruso = PlayerGameStats(player=player_caruso,minutos=48,segundos=0,puntos=10,tl_anot=1,tl_int=2,t2_anot=3,t2_int=5,t3_anot=1,t3_int=4,reb_o=1,reb_d=4,ast=5,perd=1,rec=3,tap_f=1,tap_c=0,mates=0,fc=2,fr=1,plus_minus=6,val=16)

# ---------------------------------------------------
# 3. Creamos las listas del partido
# LISTA I. Lista de jugadores en el acta del partido.
# LISTA II. Estadísticas de dichos jugadores.
# ---------------------------------------------------

j1_players = [player_shai, player_dort, player_chet, player_jalen, player_caruso]
j1_stats = [stats_shai, stats_dort, stats_chet, stats_jalen, stats_caruso]

# -------------------------------------------------------------------------------------------------------------------------
# 4. Añadimos a cada jugador sus stats en ese partido -> Estadísticas individuales de temporada.
# Lo hacemos con una función auxiliar, que usa zip() y comprueba que las stats correspondan efectivamente a cada jugador.
# -------------------------------------------------------------------------------------------------------------------------


# SE NOS OLVIDA!
#add_game_to_players(j1_players, j1_stats)

# ---------------------------------------------------
# 5. Estadísticas colectivas del equipo (TEAM)
# ---------------------------------------------------

j1_team_stats = TeamGameStats(reb_o=2,reb_d=1,perd=1,rec=0,fc=1)

# ---------------------------------------------------
# 6. Creamos el partido: objeto MyTeamGame
# ---------------------------------------------------

j1_my_team = MyTeamGame(team_name="OKC Thunder",players_list=j1_players,stats_list=j1_stats,team_stats=j1_team_stats)

# ---------------------------------------------------
# 7. Calculamos los totales del equipo
# ---------------------------------------------------

team_totals = j1_my_team.team_totals
print(team_totals)

2026-03-15 19:58:03,309 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:03,310 - INFO - __main__ - Jugador creado: Luguentz Dort (#5)
2026-03-15 19:58:03,312 - INFO - __main__ - Jugador creado: Chet Holmgren (#7)
2026-03-15 19:58:03,314 - INFO - __main__ - Jugador creado: Jalen Williams (#8)
2026-03-15 19:58:03,318 - INFO - __main__ - Jugador creado: Alex Caruso (#9)
2026-03-15 19:58:03,319 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:03,321 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:03,322 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Chet Holmgren (#7)
2026-03-15 19:58:03,323 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Jalen Williams (#8)
2026-03-15 19:58:03,325 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Alex Caruso (#

PlayerYStatsNoCorrespondientesError: Hay un problema. Revisa la lista de tus jugadores y la lista de sus estadísticas. 
Comprueba que ambas listas coinciden en el orden adecuado. 
Parece que las estadísticas de Luguentz Dort no se han añadido correctamente.

In [321]:
# EJEMPLO 2: OKC THUNDER
# EJEMPLO INCORRECTO 2-C: Uso de dorsales duplicados en un partido

"""
Nos damos cuenta del error al crear 'MyTeamGame', en el punto #5 de esta celda.
Esto es coherente. Podría darse el caso de que el mismo dorsal sea utilizado por diferentes jugadores en una temporada, por ejemplo,
tras un traspaso. El jugador que fichamos puede utilizar un dorsal que ya haya sido utilizado en la temporada.
Luego, no impedimos que haya dos jugadores en la temporada con el mismo dorsal.
Lo que impedimos es que un mismo dorsal se repita EN EL MISMO PARTIDO.

Análogamente, podría ocurrir que un jugador se cambie de dorsal durante la temporada. Esto también está permitido. No lo hemos impedido
en el código, ni lo haremos. Podemos tener dos objetos "Player" diferentes, que correspondan al mismo jugador, pero con diferente dorsal.
De este modo, podríamos analizar si el rendimiento del jugador cambia durante la temporada, tras el cambio de dorsal, etc.
"""

# 1. Creamos dos jugadores con el mismo dorsal.

player_shai = Player("Shai Gilgeous-Alexander", "2")
player_dort = Player("Luguentz Dort", "2")  # MISMO DORSAL -> ERROR.
player_chet = Player("Chet Holmgren", "7")
player_jalen = Player("Jalen Williams", "8")
player_caruso = Player("Alex Caruso", "9")

# 2. Creamos sus estadísticas en un partido
stats_shai = PlayerGameStats(player=player_shai,minutos=40,segundos=0,puntos=31,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=36)
stats_dort = PlayerGameStats(player=player_dort,minutos=40,segundos=0,puntos=14,tl_anot=2,tl_int=2,t2_anot=3,t2_int=6,t3_anot=2,t3_int=7,reb_o=1,reb_d=3,ast=2,perd=1,rec=1,tap_f=1,tap_c=0,mates=1,fc=3,fr=2,plus_minus=5,val=12)
stats_chet = PlayerGameStats(player=player_chet,minutos=40,segundos=0,puntos=19,tl_anot=4,tl_int=4,t2_anot=6,t2_int=10,t3_anot=1,t3_int=3,reb_o=3,reb_d=7,ast=2,perd=2,rec=1,tap_f=3,tap_c=1,mates=1,fc=2,fr=3,plus_minus=10,val=27)
stats_jalen = PlayerGameStats(player=player_jalen,minutos=40,segundos=0,puntos=22,tl_anot=5,tl_int=6,t2_anot=7,t2_int=12,t3_anot=1,t3_int=4,reb_o=2,reb_d=5,ast=4,perd=2,rec=2,tap_f=0,tap_c=0,mates=2,fc=1,fr=4,plus_minus=8,val=27)
stats_caruso = PlayerGameStats(player=player_caruso,minutos=40,segundos=0,puntos=10,tl_anot=1,tl_int=2,t2_anot=3,t2_int=5,t3_anot=1,t3_int=4,reb_o=1,reb_d=4,ast=5,perd=1,rec=3,tap_f=1,tap_c=0,mates=0,fc=2,fr=1,plus_minus=6,val=16)

# 3. Creamos las listas del partido

j1_players = [player_shai, player_dort, player_chet, player_jalen, player_caruso] # LISTA I. Lista de jugadores en el acta del partido.
j1_stats = [stats_shai, stats_dort, stats_chet, stats_jalen, stats_caruso] # LISTA II. Estadísticas de dichos jugadores.

# 4. Añadimos a cada jugador sus stats en ese partido -> Estadísticas individuales de temporada.
add_game_to_players(j1_players, j1_stats)

# 5. Añadimos stadísticas colectivas del equipo
j1_team_stats = TeamGameStats(reb_o=2,reb_d=1,perd=1,rec=0,fc=1)

# 6. Creamos el partido de nuestro equipo -> objeto 'MyTeamGame'
j1_my_team = MyTeamGame(team_name="OKC Thunder",players_list=j1_players,stats_list=j1_stats,team_stats=j1_team_stats)

# Veamos qué aparece un error al haber repetido el dorsal '2'.


2026-03-15 19:58:07,059 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:07,060 - INFO - __main__ - Jugador creado: Luguentz Dort (#2)
2026-03-15 19:58:07,062 - INFO - __main__ - Jugador creado: Chet Holmgren (#7)
2026-03-15 19:58:07,063 - INFO - __main__ - Jugador creado: Jalen Williams (#8)
2026-03-15 19:58:07,065 - INFO - __main__ - Jugador creado: Alex Caruso (#9)
2026-03-15 19:58:07,066 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:07,068 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Luguentz Dort (#2)
2026-03-15 19:58:07,069 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Chet Holmgren (#7)
2026-03-15 19:58:07,071 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Jalen Williams (#8)
2026-03-15 19:58:07,072 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Alex Caruso (#9)
2026-03

DorsalDuplicadoEnPartidoError: El dorsal 2 ya está en uso en este partido.

In [322]:
# EJEMPLO 2: OKC THUNDER - ¡EJEMPLO CORRECTO de "MyTeamGame"!

# ---------------------------------------------------
# 1. Creamos los jugadores
# ---------------------------------------------------

player_shai = Player("Shai Gilgeous-Alexander", "2")
player_dort = Player("Luguentz Dort", "5")
player_chet = Player("Chet Holmgren", "7")
player_jalen = Player("Jalen Williams", "8")
player_caruso = Player("Alex Caruso", "9")

# ---------------------------------------------------
# 2. Creamos las estadísticas individuales de cada jugador en el partido. Clase 'PlayerGameStats'
# ERROR -> Asignamos las estadísticas "stats_dort" al jugador "player_shai"
# ---------------------------------------------------

stats_shai = PlayerGameStats(player=player_shai,minutos=48,segundos=0,puntos=31,tl_anot=9,tl_int=10,t2_anot=8,t2_int=15,t3_anot=2,t3_int=5,reb_o=1,reb_d=4,ast=7,perd=3,rec=2,tap_f=0,tap_c=1,mates=1,fc=2,fr=8,plus_minus=12,val=36)
stats_dort = PlayerGameStats(player=player_dort,minutos=48,segundos=0,puntos=14,tl_anot=2,tl_int=2,t2_anot=3,t2_int=6,t3_anot=2,t3_int=7,reb_o=1,reb_d=3,ast=2,perd=1,rec=1,tap_f=1,tap_c=0,mates=1,fc=3,fr=2,plus_minus=5,val=12)
stats_chet = PlayerGameStats(player=player_chet,minutos=48,segundos=0,puntos=19,tl_anot=4,tl_int=4,t2_anot=6,t2_int=10,t3_anot=1,t3_int=3,reb_o=3,reb_d=7,ast=2,perd=2,rec=1,tap_f=3,tap_c=1,mates=1,fc=2,fr=3,plus_minus=10,val=27)
stats_jalen = PlayerGameStats(player=player_jalen,minutos=48,segundos=0,puntos=22,tl_anot=5,tl_int=6,t2_anot=7,t2_int=12,t3_anot=1,t3_int=4,reb_o=2,reb_d=5,ast=4,perd=2,rec=2,tap_f=0,tap_c=0,mates=2,fc=1,fr=4,plus_minus=8,val=27)
stats_caruso = PlayerGameStats(player=player_caruso,minutos=48,segundos=0,puntos=10,tl_anot=1,tl_int=2,t2_anot=3,t2_int=5,t3_anot=1,t3_int=4,reb_o=1,reb_d=4,ast=5,perd=1,rec=3,tap_f=1,tap_c=0,mates=0,fc=2,fr=1,plus_minus=6,val=16)

# ---------------------------------------------------
# 3. Creamos las listas del partido
# LISTA I. Lista de jugadores en el acta del partido.
# LISTA II. Estadísticas de dichos jugadores.
# ---------------------------------------------------

j1_players = [player_shai, player_dort, player_chet, player_jalen, player_caruso]
j1_stats = [stats_shai, stats_dort, stats_chet, stats_jalen, stats_caruso]

# -------------------------------------------------------------------------------------------------------------------------
# 4. Añadimos a cada jugador sus stats en ese partido -> Estadísticas individuales de temporada.
# Lo hacemos con una función auxiliar, que usa zip() y comprueba que las stats correspondan efectivamente a cada jugador.
# -------------------------------------------------------------------------------------------------------------------------

add_game_to_players(j1_players, j1_stats)

# ---------------------------------------------------
# 5. Estadísticas colectivas del equipo (TEAM)
# ---------------------------------------------------

j1_team_stats = TeamGameStats(reb_o=2,reb_d=1,perd=1,rec=0,fc=1)

# ---------------------------------------------------
# 6. Creamos el partido: objeto MyTeamGame
# ---------------------------------------------------

j1_my_team = MyTeamGame(team_name="OKC Thunder",players_list=j1_players,stats_list=j1_stats,team_stats=j1_team_stats)

# ---------------------------------------------------
# 7. Calculamos los totales del equipo
# ---------------------------------------------------

team_totals = j1_my_team.team_totals
print(team_totals)

# ---------------------------------------------------
# 8. Finalmente, mostramos el boxscore del partido
# ---------------------------------------------------
print("\n\n")
j1_boxscore = j1_my_team.boxscore_acb
j1_boxscore

2026-03-15 19:58:11,535 - INFO - __main__ - Jugador creado: Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:11,537 - INFO - __main__ - Jugador creado: Luguentz Dort (#5)
2026-03-15 19:58:11,539 - INFO - __main__ - Jugador creado: Chet Holmgren (#7)
2026-03-15 19:58:11,540 - INFO - __main__ - Jugador creado: Jalen Williams (#8)
2026-03-15 19:58:11,543 - INFO - __main__ - Jugador creado: Alex Caruso (#9)
2026-03-15 19:58:11,546 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Shai Gilgeous-Alexander (#2)
2026-03-15 19:58:11,548 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Luguentz Dort (#5)
2026-03-15 19:58:11,549 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Chet Holmgren (#7)
2026-03-15 19:58:11,552 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Jalen Williams (#8)
2026-03-15 19:58:11,553 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Alex Caruso (#9)
2026-03

{'jugador': 'Total (OKC Thunder)', 'dorsal': '', 'min_played': 240.0, 'puntos': 96, 'tl_anot': 21, 'tl_int': 24, 'tl_pct': 87.5, 't2_anot': 27, 't2_int': 48, 't2_pct': 56.25, 't3_anot': 7, 't3_int': 23, 't3_pct': 30.434782608695656, 'reb_o': 10, 'reb_d': 24, 'reb_tot': 34, 'ast': 20, 'perd': 10, 'rec': 9, 'tap_f': 5, 'tap_c': 2, 'mates': 5, 'fc': 11, 'fr': 18, 'plus_minus': 41, 'val': 119}





,DORSAL,PLAYER,PTS,MIN,T2_Anot,T2_Int,T2%,T3_Anot,T3_Int,T3%,...,AST,TO,STL,BLK,BLK_Rec,DUNK,FC,FR,+/-,VAL
0,2,Shai Gilgeous-Alexander,31,48:00,8,15,53.33,2,5,40.0,...,7,3,2,0,1,1,2,8,12,36
1,5,Luguentz Dort,14,48:00,3,6,50.0,2,7,28.57,...,2,1,1,1,0,1,3,2,5,12
2,7,Chet Holmgren,19,48:00,6,10,60.0,1,3,33.33,...,2,2,1,3,1,1,2,3,10,27
3,8,Jalen Williams,22,48:00,7,12,58.33,1,4,25.0,...,4,2,2,0,0,2,1,4,8,27
4,9,Alex Caruso,10,48:00,3,5,60.0,1,4,25.0,...,5,1,3,1,0,0,2,1,6,16
5,,Equipo,0,0:00,0,0,0.0,0,0,0.0,...,0,1,0,0,0,0,1,0,0,1
6,,,,,,,,,,,...,,,,,,,,,,
7,,Total (OKC Thunder),96,240:00,27,48,56.25,7,23,30.43,...,20,10,9,5,2,5,11,18,41,119


---
---
---


### **CLASE 'OpponentGameStats'**


In [323]:
@dataclass
class OpponentTeamGame:
    """
    Estadísticas agregadas del equipo rival en un partido.
    Compatible con el formato PlayerGameStats/TeamGameStats para crear boxscores.
    """

    team_name: str
    dorsal: str

    min_played: int

    puntos: int

    tl_anot: int
    tl_int: int

    t2_anot: int
    t2_int: int

    t3_anot: int
    t3_int: int

    reb_o: int
    reb_d: int

    ast: int
    perd: int
    rec: int

    tap_f: int
    tap_c: int

    mates: int

    fc: int
    fr: int

    plus_minus: int
    val: int

    def __post_init__(self):
        self._validate_stats()
        logger.info(f"'OpponentTeamGame': Stats cargadas correctamente para el equipo rival ({self.team_name})")

    # ==========================================================
    # VALIDACIONES
    # ==========================================================

    def _validate_stats(self):
        """
        Valida coherencia estadística del boxscore rival.
        """

        # ----------------------------------------------------------------
        # 1. VALIDACIÓN GENERAL DATOS -> VALORES ENTEROS Y NO NEGATIVOS
        # ----------------------------------------------------------------

        integer_stats = [self.plus_minus] # El +- es la única estadística que puede ser negativa. -> Solo comprobamos que sea entero.
        non_negative_stats = [self.min_played,self.puntos,self.tl_anot, self.tl_int,self.t2_anot, self.t2_int,self.t3_anot, self.t3_int,self.reb_o, self.reb_d,self.ast, self.perd, self.rec,self.tap_f, self.tap_c,self.mates,self.fc, self.fr, self.val]

        for stat in (non_negative_stats + integer_stats):
            if not isinstance(stat, int):
              logger.error("Valor no entero detectado en estadísticas. Equipo rival")
              raise ValueError(f"Equipo rival: Las estadísticas deben ser números enteros. Revisa el '{stat}'.")

        for stat in non_negative_stats:
            if stat < 0:
              logger.error("Valor negativo detectado en estadísticas. Equipo rival.")
              raise ValueError(f"Equipo rival: Las estadísticas no pueden ser negativas. Revisa el '{stat}'.")

        # --------------------------------------------------
        # 2. TIROS COHERENTES (ANOTADOS <= INTENTADOS)
        # --------------------------------------------------

        if (self.tl_anot > self.tl_int):
            logger.error("Se detectó un error en los tiros libres. Equipo rival.")
            raise InvalidStatError("TL anotados no pueden superar TL intentados. Equipo rival.")

        if (self.t2_anot > self.t2_int):
            logger.error("Se detectó un error en los tiros de dos puntos del rival.")
            raise InvalidStatError("T2 anotados no pueden superar T2 intentados. Equipo rival")

        if (self.t3_anot > self.t3_int):
            logger.error("Se detectó un error en los tiros de tres puntos. Equipo rival.")
            raise InvalidStatError("T3 anotados no pueden superar T3 intentados. Equipo rival.")

        # --------------------------------------------------
        # 3. PUNTOS COHERENTES CON TIROS ANOTADOS
        # --------------------------------------------------

        puntos_calculados = (self.tl_anot + 2 * self.t2_anot + 3 * self.t3_anot)

        if self.puntos != puntos_calculados:
            logger.error("Los puntos no coinciden con los tiros anotados. Equipo rival.")
            raise InvalidStatError(f"Puntos incorrectos para el equipo rival: {self.team_name}."
                                   f"\nPuntos introducidos: {self.puntos}"
                                   f"\nDe acuerdo con los tiros anotados, los puntos anotados deberían ser: {puntos_calculados}.")

        # ----------------------------------------------------------------
        # 4. TAPONES RECIBIDOS COHERENTES CON TIROS DE CAMPO INTENTADOS.
        # ----------------------------------------------------------------

        tiros_campo_int = self.t2_int + self.t3_int

        if self.tap_c > tiros_campo_int:
            logger.error("Se detectó una inconsistencia entre tiros intentados y tapones recibidos. Equipo rival.")
            raise InvalidStatError("No se pueden recibir más tapones que tiros de campo intentados. Equipo rival.")

        # --------------------------------------------------
        # 5. VALORACIÓN ACB COHERENTE CON RESTO DE STATS
        # --------------------------------------------------

        valoracion_calculada = valoracion_acb(self.puntos, self.reb_o, self.reb_d, self.ast, self.rec, self.tap_f, self.fr, self.tl_anot, self.tl_int, self.t2_anot, self.t2_int, self.t3_anot, self.t3_int, self.perd, self.tap_c, self.fc)

        if self.val != valoracion_calculada:
            logger.error("Valoración ACB incorrecta.")
            raise InvalidStatError(f"Valoración incorrecta para el equipo {self.team_name}."
                                   f"\nValoración introducida: {self.val}"
                                   f"\nDe acuerdo con las estadísticas introducidas, la valoración debería ser: {valoracion_calculada}.")

    # ==========================================================
    # MIS MÉTODOS -> ESTADÍSTICA CONVENCIONAL (Propiedades)
    # Necesitamos rebotes totales y % de tiro para boxscore.
    # ==========================================================

    @property
    def reb_tot(self):
        return reb_tot(self.reb_o, self.reb_d)

    @property
    def tl_pct(self):
        return tl_pct(self.tl_anot, self.tl_int)

    @property
    def t2_pct(self):
        return t2_pct(self.t2_anot, self.t2_int)

    @property
    def t3_pct(self):
        return t3_pct(self.t3_anot, self.t3_int)

    # ==========================================================
    # MIS MÉTODOS -> CONVERTIR STATS EN DICCIONARIO
    # ==========================================================

    @property
    def to_dict(self):

        return {

            "jugador": self.team_name,
            "dorsal": self.dorsal,

            "min_played": self.min_played,

            "puntos": self.puntos,

            "tl_anot": self.tl_anot,
            "tl_int": self.tl_int,
            "tl_pct": self.tl_pct,

            "t2_anot": self.t2_anot,
            "t2_int": self.t2_int,
            "t2_pct": self.t2_pct,

            "t3_anot": self.t3_anot,
            "t3_int": self.t3_int,
            "t3_pct": self.t3_pct,

            "reb_o": self.reb_o,
            "reb_d": self.reb_d,
            "reb_tot": self.reb_tot,

            "ast": self.ast,
            "perd": self.perd,
            "rec": self.rec,

            "tap_f": self.tap_f,
            "tap_c": self.tap_c,

            "mates": self.mates,

            "fc": self.fc,
            "fr": self.fr,

            "plus_minus": self.plus_minus,

            "val": self.val
        }

In [324]:
# EJEMPLO 1 - EJEMPLO INCORRECTO DE 'OpponentTeamGame'
stats_spurs = OpponentTeamGame("San Antonio Spurs","",240,125,17,25,24,38,20,47,9,35,27,11,3,2,4,10,19,18,-9.5,136)

2026-03-15 19:58:22,230 - ERROR - __main__ - Valor no entero detectado en estadísticas. Equipo rival


ValueError: Equipo rival: Las estadísticas deben ser números enteros. Revisa el '-9.5'.

In [325]:
# EJEMPLO 2 - EJEMPLO CORRECTO DE 'OpponentTeamGame'
stats_spurs = OpponentTeamGame("San Antonio Spurs","",240,125,17,25,24,38,20,47,9,35,27,11,3,2,4,10,19,18,-9,136)

2026-03-15 19:58:24,560 - INFO - __main__ - 'OpponentTeamGame': Stats cargadas correctamente para el equipo rival (San Antonio Spurs)


In [326]:
# EJEMPLO 3 - EJEMPLO CORRECTO DE 'OpponentTeamGame'
stats_usa = OpponentTeamGame("USA","",200,99,20,25,20,38,13,33,9,35,27,11,3,2,4,10,19,18,-60,116)

2026-03-15 19:58:26,057 - INFO - __main__ - 'OpponentTeamGame': Stats cargadas correctamente para el equipo rival (USA)


### **CLASE 'Game'**

In [340]:
#NUEVO
@dataclass
class Game:
    """
    Representa un partido completo.

    Integra:
    - MyTeamGame (mi equipo)
    - OpponentTeamGame (equipo rival)

    Permite:
    - Construir boxscore completo del partido
    - Calcular estadísticas avanzadas del partido
    """

    jornada: int
    fecha: datetime.datetime
    lugar: str

    my_team_game: "MyTeamGame"
    opponent_team_game: "OpponentTeamGame"


    def __post_init__(self):
        self._validate_game_info()
        my_points = self.my_team_game.team_totals["puntos"]
        opp_points = self.opponent_team_game.puntos

        logger.info(
            f"\n\n'Game': Información cargada correctamente \nJornada: {self.jornada} | Fecha: {self.fecha.strftime("%d/%m/%Y")} | Lugar: {self.lugar} | \n"
            f"{self.my_team_game.team_name} vs {self.opponent_team_game.team_name}: {my_points} - {opp_points} "
        )


    def _validate_game_info(self):
        """
        Valida coherencia de los datos del partido: número jornada, fecha, lugar, etc.
        """

        if not isinstance(self.jornada, int):
              logger.error(f"Valor no entero detectado en jornada: '{self.jornada}'")
              raise ValueError(f"La jornada debe ser un números entero y positivo. Revisa el '{self.jornada}'.")

        if self.jornada < 1:
            logger.error(f"Jornada inválida: {self.jornada}")
            raise ValueError("La jornada debe ser un entero mayor o igual a 1.")

        if not isinstance(self.fecha, datetime.datetime):
            logger.error("Fecha inválida.")
            raise ValueError("La fecha debe tener formato datetime. Ejemplo: 'datetime.datetime(2019/4/14)' .")

        if not self.my_team_game:
            logger.error("No se proporcionó 'MyTeamGame'.")
            raise ValueError("Se requiere 'MyTeamGame'.")

        if not self.opponent_team_game:
            logger.error("No se proporcionó 'OpponentTeamGame'.")
            raise ValueError("Se requiere 'OpponentTeamGame'.")


    # ==========================================================
    # RESUMEN DEL PARTIDO
    # ==========================================================

    @property
    def game_info(self):
        """
        Resumen básico del partido.
        """
        my_points = self.my_team_game.team_totals["puntos"]
        opp_points = self.opponent_team_game.puntos

        return {
            "JORNADA": self.jornada,
            "FECHA": self.fecha.strftime("%d/%m/%Y"),
            "LUGAR": self.lugar,
            "MI_EQUIPO": self.my_team_game.team_name,
            "RIVAL": self.opponent_team_game.team_name,
            "RESULTADO": f"{my_points}-{opp_points}"
        }

    # ==========================================================
    # BOXSCORE COMPLETO DEL PARTIDO
    # ==========================================================

    @property
    def game_boxscore(self):
        """
        Construye el boxscore completo del partido.

        Incluye:
        - jugadores de mi equipo
        - TEAM
        - totales
        - rival
        """

        # boxscore de mi equipo
        my_df = self.my_team_game.boxscore.copy()

        # stats del rival
        opponent_row = pd.DataFrame([self.opponent_team_game.to_dict])

        # separador visual
        separator = pd.DataFrame([{col: "" for col in my_df.columns}])

        # concatenar
        game_df = pd.concat(
            [my_df, separator, opponent_row],
            ignore_index=True
        )

        return game_df

    # ==========================================================
    # BOXSCORE FORMATO ACB
    # ==========================================================

    @property
    def game_boxscore_acb(self):
        """
        Versión visual del boxscore del partido
        usando el formato ACB.
        """

        display_df = self.game_boxscore.copy()

        column_names = {
            "jugador": "PLAYER",
            "dorsal": "DORSAL",
            "puntos": "PTS",
            "min_played": "MIN",
            "tl_anot": "TL_Anot",
            "tl_int": "TL_Int",
            "tl_pct": "TL%",
            "t2_anot": "T2_Anot",
            "t2_int": "T2_Int",
            "t2_pct": "T2%",
            "t3_anot": "T3_Anot",
            "t3_int": "T3_Int",
            "t3_pct": "T3%",
            "reb_d": "DR",
            "reb_o": "OR",
            "reb_tot": "TR",
            "ast": "AST",
            "perd": "TO",
            "rec": "STL",
            "tap_f": "BLK",
            "tap_c": "BLK_Rec",
            "mates": "DUNK",
            "fc": "FC",
            "fr": "FR",
            "plus_minus": "+/-",
            "val": "VAL"
        }

        display_df.rename(columns=column_names, inplace=True)

        ordered_cols = [
            "DORSAL",
            "PLAYER",
            "PTS",
            "MIN",
            "T2_Anot","T2_Int","T2%",
            "T3_Anot","T3_Int","T3%",
            "TL_Anot","TL_Int","TL%",
            "DR","OR","TR",
            "AST","TO","STL",
            "BLK","BLK_Rec",
            "DUNK",
            "FC","FR",
            "+/-","VAL"
        ]

        display_df = display_df[ordered_cols]

        display_df["MIN"] = display_df["MIN"].apply(format_minutes)

        return display_df.round(2)


    # ==========================================================
    # ESTADÍSTICA AVANZADA - JUGADORES
    # ==========================================================

    @property
    def advanced_players(self):

        players_df = self.my_team_game.dataframe_only_players
        team_totals = self.my_team_game.team_totals
        opponent_totals = self.opponent_team_game.to_dict

        df = calculate_advanced_players(players_df,team_totals,opponent_totals)

        return df

    # ==========================================================
    # ESTADÍSTICA AVANZADA - EQUIPO
    # ==========================================================

    @property
    def advanced_team(self):

        team_totals = self.my_team_game.team_totals
        opponent_totals = self.opponent_team_game.to_dict

        return calculate_advanced_team(team_totals,opponent_totals)

    # ==========================================================
    # ESTADÍSTICA AVANZADA - EQUIPO CONTRARIO
    # Basta intercambiar orden al llamar a la función que lo calcula
    # ==========================================================

    @property
    def advanced_opponent(self):

        team_totals = self.my_team_game.team_totals
        opponent_totals = self.opponent_team_game.to_dict

        return calculate_advanced_team(opponent_totals,team_totals)

In [341]:
# EJEMPLO 1 - EJEMPLO INCORRECTO DE 'Game'
j1 = Game(-5,datetime.datetime(2008,8,24),"Beijing",spain_game,stats_usa)
# Falla porque 'jornada' debe ser un número entero positivo

2026-03-15 20:01:00,853 - ERROR - __main__ - Jornada inválida: -5


ValueError: La jornada debe ser un entero mayor o igual a 1.

In [329]:
# EJEMPLO 2 - EJEMPLO INCORRECTO DE 'Game'
j1 = Game(1,"2008/8/24","Beijing",spain_game,stats_usa)
# Falla porque 'fecha' debe tener exactamente formato datetime: "datetime.datetime(año,mes,día)"
# No puede ser un string, ni ninguna otra cosa

2026-03-15 19:58:39,469 - ERROR - __main__ - Fecha inválida.


ValueError: La fecha debe tener formato datetime. Ejemplo: 'datetime.datetime(2019/4/14)' .

In [330]:
# EJEMPLO 3 - EJEMPLO INCORRECTO DE 'Game'
j1 = Game(1,"2008/8/24",Beijing,spain_game,stats_usa)
# Falla porque 'lugar' debe  ser un string.

NameError: name 'Beijing' is not defined

In [342]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
'''
Reutilizamos:
-Ejemplo de "MyTeamGame" de la selección española.
-Ejemplo de "OpponentTeamGame" de la selección de USA.

Nótese que 'jornada' debe ser un entero positivo, 'fecha' debe tener formato datetime y "lugar" debe ser un string.
'''

j1 = Game(1,datetime.datetime(2008,8,24),"Beijing",spain_game,stats_usa)

2026-03-15 20:01:12,073 - INFO - __main__ - 

'Game': Información cargada correctamente 
Jornada: 1 | Fecha: 24/08/2008 | Lugar: Beijing | 
España vs USA: 105 - 99 


In [332]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
# INFORMACIÓN DEL PARTIDO
j1.game_info

{'JORNADA': 1,
 'FECHA': '24/08/2008',
 'LUGAR': 'Beijing',
 'MI_EQUIPO': 'España',
 'RIVAL': 'USA',
 'RESULTADO': '105-99'}

In [333]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
# BOXSCORE (FORMATO INTERNO)
j1_boxscore_completo = j1.game_boxscore
j1_boxscore_completo

,jugador,dorsal,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,t2_pct,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
0,Pau Gasol,4,18.0,16,4,5,80.0,6,10,60.0,...,2,2,1,2,1,1,3,4,9,23
1,Rudy Fernandez,5,18.0,11,4,4,100.0,2,4,50.0,...,2,1,2,1,0,1,2,3,6,16
2,Juan Carlos Navarro,7,22.0,14,4,4,100.0,2,4,50.0,...,2,1,1,0,0,0,1,4,7,15
3,Jose Manuel Calderon,8,21.5,10,2,2,100.0,1,3,33.33,...,4,1,1,0,0,0,1,2,6,13
4,Ricky Rubio,9,22.5,8,2,2,100.0,3,5,60.0,...,7,2,2,0,0,0,2,3,5,16
5,Sergio Rodriguez,13,22.0,7,1,2,50.0,3,5,60.0,...,6,2,1,0,0,0,1,1,4,9
6,Serge Ibaka,14,18.0,9,1,2,50.0,4,6,66.67,...,0,1,1,3,0,2,2,2,5,15
7,Jorge Garbajosa,15,18.0,6,0,0,0.0,3,5,60.0,...,1,1,0,0,0,0,2,1,3,5
8,Sergio Llull,23,22.0,11,2,2,100.0,3,6,50.0,...,3,1,1,0,0,0,2,3,8,12
9,Marc Gasol,33,18.0,13,3,4,75.0,5,8,62.5,...,3,1,1,1,1,1,2,3,7,19


In [334]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
# BOXSCORE (FORMATO ACB)
j1_boxscore_acb = j1.game_boxscore_acb
j1_boxscore_acb

,DORSAL,PLAYER,PTS,MIN,T2_Anot,T2_Int,T2%,T3_Anot,T3_Int,T3%,...,AST,TO,STL,BLK,BLK_Rec,DUNK,FC,FR,+/-,VAL
0,4,Pau Gasol,16,18:00,6,10,60.0,0,0,0.0,...,2,2,1,2,1,1,3,4,9,23
1,5,Rudy Fernandez,11,18:00,2,4,50.0,1,3,33.33,...,2,1,2,1,0,1,2,3,6,16
2,7,Juan Carlos Navarro,14,22:00,2,4,50.0,2,5,40.0,...,2,1,1,0,0,0,1,4,7,15
3,8,Jose Manuel Calderon,10,21:30,1,3,33.33,2,4,50.0,...,4,1,1,0,0,0,1,2,6,13
4,9,Ricky Rubio,8,22:30,3,5,60.0,0,2,0.0,...,7,2,2,0,0,0,2,3,5,16
5,13,Sergio Rodriguez,7,22:00,3,5,60.0,0,2,0.0,...,6,2,1,0,0,0,1,1,4,9
6,14,Serge Ibaka,9,18:00,4,6,66.67,0,0,0.0,...,0,1,1,3,0,2,2,2,5,15
7,15,Jorge Garbajosa,6,18:00,3,5,60.0,0,2,0.0,...,1,1,0,0,0,0,2,1,3,5
8,23,Sergio Llull,11,22:00,3,6,50.0,1,3,33.33,...,3,1,1,0,0,0,2,3,8,12
9,33,Marc Gasol,13,18:00,5,8,62.5,0,1,0.0,...,3,1,1,1,1,1,2,3,7,19


In [343]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
# CALCULADORA ESTADÍSTICA AVANZADA - JUGADORES
prueba_players = j1.advanced_players
prueba_players


,jugador,dorsal,POSS_USED,OER,EFG%,TOV%,OREB%,FT_RATE,DREB%,TS%,AST/TOV,AST_RATIO%,AST%,STL%,BLK%,USG%,USG_TOT%
8,Pau Gasol,4,14.20,1.13,60.0,12.35,14.49,0.5,31.75,65.57,1.0,12.35,7.97,2.65,11.7,30.38,13.67
5,Rudy Fernandez,5,9.76,1.13,50.0,8.5,4.83,0.57,15.87,62.79,2.0,17.01,7.12,5.29,5.85,20.88,9.40
4,Juan Carlos Navarro,7,11.76,1.19,55.56,7.27,0.0,0.44,4.33,65.06,2.0,14.53,5.9,2.16,0.0,20.58,11.32
1,Jose Manuel Calderon,8,8.88,1.13,57.14,7.76,0.0,0.29,8.86,63.45,4.0,31.06,11.45,2.21,0.0,15.90,8.55
0,Ricky Rubio,9,9.88,0.81,42.86,11.85,3.86,0.29,12.7,50.76,3.5,41.47,18.98,4.23,0.0,16.91,9.51
2,Sergio Rodriguez,13,9.88,0.71,42.86,12.59,0.0,0.29,8.66,44.42,3.0,37.78,16.71,2.16,0.0,17.29,9.51
6,Serge Ibaka,14,7.88,1.14,66.67,12.69,9.66,0.33,21.16,65.41,0.0,0.0,0.0,2.65,17.54,16.86,7.59
7,Jorge Garbajosa,15,8.00,0.75,42.86,11.11,4.83,0.0,15.87,42.86,1.0,11.11,3.56,0.0,0.0,17.11,7.70
3,Sergio Llull,23,10.88,1.01,50.0,7.2,0.0,0.22,8.66,55.67,3.0,21.61,8.85,2.16,0.0,19.04,10.47
9,Marc Gasol,33,11.76,1.11,55.56,6.78,9.66,0.44,26.46,60.41,3.0,20.33,11.49,2.65,5.85,25.16,11.32


In [344]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
# CALCULADORA ESTADÍSTICA AVANZADA - EQUIPO
prueba_team = j1.advanced_team
prueba_team

,jugador,POSS,OER,DER,NET,EFG%,TOV%,OREB%,FT_RATE,DREB%,TS%,AST/TOV,AST_RATIO%,AST%,STL%,BLK%,PACE_ACB,PACE_NBA
0,Total (España),92.88,113.05,117.86,-4.81,52.56,13.48,23.91,0.35,78.57,58.41,2.14,28.88,38.46,13.1,18.42,88.44,106.13


In [345]:
# EJEMPLO 4 - EJEMPLO CORRECTO DE 'Game'
# CALCULADORA ESTADÍSTICA AVANZADA - EQUIPO CONTRARIO

prueba_opp = j1.advanced_opponent
prueba_opp

,jugador,POSS,OER,DER,NET,EFG%,TOV%,OREB%,FT_RATE,DREB%,TS%,AST/TOV,AST_RATIO%,AST%,STL%,BLK%,PACE_ACB,PACE_NBA
0,USA,84.0,117.86,113.05,4.81,55.63,11.83,21.43,0.35,76.09,60.37,2.45,29.03,38.03,3.23,3.57,88.44,106.13


### **CLASE 'Season'**

In [346]:
# ==========================================================
# CLASE SEASON
# ==========================================================

@dataclass
class Season:

    competition: str
    season: str
    games: list

    def __post_init__(self):
        self.validate_all()
        logger.info(
            f"\n'Season' cargada correctamente\nCompetición: {self.competition} | Temporada: {self.season} | Nº de partidos: {len(self.games)}"
        )

    # ==========================================================
    # VALIDACIONES
    # ==========================================================

    def validate_competition(self):

        if not isinstance(self.competition, str):
            raise ValueError("La competición debe ser un string.")

        texto = self.competition.strip()

        patron = r"[A-Za-z0-9ÁÉÍÓÚÜáéíóúüñÑªº \-()/.,]+"

        if not re.fullmatch(patron, texto):
            raise ValueError("Nombre de competición inválido.")

    def validate_season(self):

        if not isinstance(self.season, str):
            raise ValueError("La temporada debe ser un string.")

        patron = r"\d{4}(-\d{4})?"

        if not re.fullmatch(patron, self.season):
            raise ValueError("Formato de temporada inválido.")

        if "-" in self.season:
            inicio, fin = map(int, self.season.split("-"))
            if fin != inicio + 1:
                raise ValueError("Los años deben ser consecutivos.")

    def validate_games(self):

        if not isinstance(self.games, list):
            raise ValueError("games debe ser una lista.")

        for g in self.games:
            if not isinstance(g, Game):
                raise ValueError("Todos los elementos de games deben ser objetos Game.")

    def validate_all(self):

        self.validate_competition()
        self.validate_season()
        self.validate_games()

    # ==========================================================
    # BOXSCORE TEMPORADA (GAME LOG)
    # ==========================================================

    @property
    def season_boxscore(self):

        dfs = []

        for g in self.games:

            df = g.my_team_game.dataframe_only_players.copy()

            df["game"] = g.jornada
            df["rival"] = g.opponent_team_game.team_name
            df["fecha"] = g.fecha

            dfs.append(df)

        return pd.concat(dfs, ignore_index=True)

    # ==========================================================
    # TOTALES TEMPORADA (JUGADORES + EQUIPO)
    # ==========================================================


    @property
    def season_totals(self):

        df = self.season_boxscore

        numeric_cols = df.select_dtypes(include="number").columns
        numeric_cols = [c for c in numeric_cols if c not in ["game","tl_pct","t2_pct","t3_pct"]]

        players = df.groupby(["jugador","dorsal"])[numeric_cols].sum().reset_index()

        # partidos jugados
        games_played = (
            self.season_boxscore
            .groupby(["jugador","dorsal"])
            .size()
            .reset_index(name="games_played")
        )

        players = players.merge(games_played,on=["jugador","dorsal"])

        # recalcular porcentajes
        players["tl_pct"] = (players["tl_anot"]/players["tl_int"]*100).round(2)
        players["t2_pct"] = (players["t2_anot"]/players["t2_int"]*100).round(2)
        players["t3_pct"] = (players["t3_anot"]/players["t3_int"]*100).round(2)

        players[["tl_pct","t2_pct","t3_pct"]] = players[["tl_pct","t2_pct","t3_pct"]].fillna(0)

        # ordenar por dorsal
        players["_dorsal_sort"] = players["dorsal"].apply(lambda x:int(x) if str(x).isdigit() else 999)
        players = players.sort_values("_dorsal_sort")
        players.drop(columns="_dorsal_sort",inplace=True)

        # totales equipo
        team = pd.Series(self.team_totals)

        team["games_played"] = len(self.games)

        team["tl_pct"] = round(team["tl_anot"]/team["tl_int"]*100,2) if team["tl_int"] else 0
        team["t2_pct"] = round(team["t2_anot"]/team["t2_int"]*100,2) if team["t2_int"] else 0
        team["t3_pct"] = round(team["t3_anot"]/team["t3_int"]*100,2) if team["t3_int"] else 0

        team_row = pd.DataFrame([team])

        separator = pd.DataFrame([{col:"" for col in players.columns}])

        totals = pd.concat([players,separator,team_row],ignore_index=True)

        ordered_cols = [
            "jugador","dorsal","games_played",
            "min_played","puntos",
            "tl_anot","tl_int","tl_pct",
            "t2_anot","t2_int","t2_pct",
            "t3_anot","t3_int","t3_pct",
            "reb_o","reb_d","reb_tot",
            "ast","perd","rec",
            "tap_f","tap_c",
            "mates",
            "fc","fr",
            "plus_minus","val"
        ]

        totals = totals[ordered_cols]

        return totals



    # ==========================================================
    # PROMEDIOS TEMPORADA (DIVIDIENDO POR PARTIDOS JUGADOS)
    # ==========================================================

    @property
    def season_averages(self):

        df = self.season_boxscore

        numeric_cols = df.select_dtypes(include="number").columns
        numeric_cols = [c for c in numeric_cols if c not in ["game","tl_pct","t2_pct","t3_pct"]]

        players = df.groupby(["jugador","dorsal"])[numeric_cols].sum().reset_index()

        # partidos jugados
        games_played = (self.season_boxscore.groupby(["jugador","dorsal"]).size().reset_index(name="games_played"))
        players = players.merge(games_played,on=["jugador","dorsal"])

        # dividir stats
        cols_divisibles= ["min_played","puntos","tl_anot","tl_int","t2_anot","t2_int","t3_anot","t3_int","reb_o","reb_d","reb_tot","ast","perd","rec","tap_f","tap_c","mates","fc","fr","plus_minus","val"]

        for stat in cols_divisibles:
            players[stat] = players[stat] / players["games_played"]

        # recalcular porcentajes
        players["tl_pct"] = (players["tl_anot"]/players["tl_int"]*100).round(2)
        players["t2_pct"] = (players["t2_anot"]/players["t2_int"]*100).round(2)
        players["t3_pct"] = (players["t3_anot"]/players["t3_int"]*100).round(2)

        players[["tl_pct","t2_pct","t3_pct"]] = players[["tl_pct","t2_pct","t3_pct"]].fillna(0)

        # ordenar por dorsal
        players["_dorsal_sort"] = players["dorsal"].apply(lambda x:int(x) if str(x).isdigit() else 999)
        players = players.sort_values("_dorsal_sort")
        players.drop(columns="_dorsal_sort",inplace=True)

        ordered_cols = [
            "jugador","dorsal","games_played",
            "min_played","puntos",
            "tl_anot","tl_int","tl_pct",
            "t2_anot","t2_int","t2_pct",
            "t3_anot","t3_int","t3_pct",
            "reb_o","reb_d","reb_tot",
            "ast","perd","rec",
            "tap_f","tap_c",
            "mates",
            "fc","fr",
            "plus_minus","val"
        ]

        players = players[ordered_cols]

        return players

    # ==========================================================
    # TOTALES EQUIPO TEMPORADA
    # ==========================================================

    @property
    def team_totals(self):

        dfs = []

        for g in self.games:
            dfs.append(pd.DataFrame([g.my_team_game.team_totals]))

        df = pd.concat(dfs)

        numeric_cols = df.select_dtypes(include="number").columns

        totals = df[numeric_cols].sum()

        totals["jugador"] = "Totales"
        totals["dorsal"] = ""

        return totals.to_dict()

    # ==========================================================
    # TOTALES RIVAL TEMPORADA
    # ==========================================================

    @property
    def opponent_totals(self):

        dfs = []

        for g in self.games:
            dfs.append(pd.DataFrame([g.opponent_team_game.to_dict]))

        df = pd.concat(dfs)

        numeric_cols = df.select_dtypes(include="number").columns

        totals = df[numeric_cols].sum()

        return totals.to_dict()

    # ==========================================================
    # ADVANCED PLAYERS TEMPORADA
    # ==========================================================

    @property
    def season_advanced_players(self):

        df = self.season_totals.copy()
        df = df[df["jugador"]!=""]
        players = df[df["jugador"]!="Totales"].copy()
        df_advanced = calculate_advanced_players(players,self.team_totals,self.opponent_totals)

        # Aquí no hacen falta las líneas del "separator", porque yo ya había añadido
        # una columna "TOTALES" en la función calculate_advanced_players, sin espacio entre Jugadores y "TOTALES"

        #separator = pd.DataFrame([{col:"" for col in adv.columns}])
        #team_row = pd.DataFrame([{col:"" for col in adv.columns}])
        #result = pd.concat([adv,separator,team_row],ignore_index=True)

        return df_advanced

    # ==========================================================
    # ADVANCED TEAM TEMPORADA
    # ==========================================================

    @property
    def season_advanced_team(self):

        df_adv = calculate_advanced_team(self.team_totals,self.opponent_totals)

        #La calculadora nos devuelve el número de posesiones totales.
        #Añado dos columnas adicionales:
        df_adv["GAMES_PLAYED"] = len(self.games)
        df_adv["POSS_per_game"] = df_adv["POSS"] / len(self.games)

        return df_adv.round(2)

    # =================================================================
    # LEADERBOARDS -> RANKING DE JUGADORES POR ESTADÍSTICA CONVENCIONAL
    # =================================================================

    def leaderboard_conv(self,stat="puntos",top=5,ascending=False):

        df = self.season_averages.copy()

        if stat not in df.columns:
            raise ValueError(f"La estadística '{stat}' no existe.")

        df = df.sort_values(stat,ascending=ascending)
        cols=["jugador","dorsal",stat]

        return df[cols].head(top)


    # ==============================================================
    # LEADERBOARDS -> RANKING DE JUGADORES POR ESTADÍSTICA AVANZADA
    # ==============================================================

    def leaderboard_adv(self,stat="OER",top=5,ascending=False):

        df = self.season_advanced_players.copy()

        # ELIMINAMOS LA FILA DE "TOTALES" -> Solo queremos comparar jugadores entre sí.
        # Si la dejaramos, nos darían errores.

        df = df[df["jugador"]!="TOTALES"]
        if stat not in df.columns:
            raise ValueError(f"La estadística '{stat}' no existe.")

        df = df.sort_values(stat,ascending=ascending)
        cols=["jugador","dorsal",stat]

        return df[cols].head(top)

---
---
---

**=====================================**
## **CAPA 3 — APLICACIÓN**
**=====================================**

### **EJEMPLO FINAL: Clase 'Calculator'**

-DECISIÓN IMPORTANTE DE DISEÑO: He decidido no pedir los datos como input. Introducir todos los datos para jugadores, stats del partido, stats de equipo, stats de rivales, y repetir todo eso para cada jornada, sería literalmente un infierno para probar la calculadora. Además, al realizar las validaciones, podrían ocurrir fallos del usuario al imputar valores, haciendo aún más largo el proceso. Se tardarían horas en probar la calculadora. No es eficiente para su uso.

-En su lugar, el usuario debe cargar los datos, siguiendo los ejemplos que se han mostrado. Las etiquetas de las variables son explicativas, y todo el código está comentado, por lo que no habrá problema al realizar esto. A partir de un ojeto 'season' ya construido, se corre la aplicación de "Calculator". Esta engloba todas las funcionalidades que hemos implementado, permitiendo calcular totales de temporada, promedios, estadística avanzada...

- Vamos a retomar el ejemplo de la selección española, que ya habíamos construido, con el objeto "season1". Añado el código necesario para ejecutar la calculadora.



### **Clase 'Season' - EJEMPLO COMPLETO**

Una vez construidas todas las clases, vamos con el ejemplo final de la clase 'Season'.

Retomamos el ejemplo de la selección española, y creamos otras dos jornadas. Con esas tres jornadas, creamos un ejemplo de clase 'Season', incluyendo el uso de sus métodos: estadística promedio, acumulada y estadística avanzada de la temporada, tanto para jugadores como para el equipo.

Sin más comentarios, el código final:

In [347]:
# EJEMPLO FINAL: SELECCIÓN ESPAÑOLA - EJEMPLO CORRECTO de "SEASON"

# ==========================================
# CREAR JUGADORES: 'Player'
# ==========================================

ricky = Player("Ricky Rubio", "9")
calderon = Player("Jose Manuel Calderon", "8")
chacho = Player("Sergio Rodriguez", "13")
llull = Player("Sergio Llull", "23")
navarro = Player("Juan Carlos Navarro", "7")
rudy = Player("Rudy Fernandez", "5")
ibaka = Player("Serge Ibaka", "14")
garbajosa = Player("Jorge Garbajosa", "15")
pau = Player("Pau Gasol", "4")
marc = Player("Marc Gasol", "33")

# ==========================================================
# JORNADA 1 — ESPAÑA vs USA
# ==========================================================

ricky_stats_j1 = PlayerGameStats(player=ricky,minutos=22,segundos=30,puntos=8,tl_anot=2,tl_int=2,t2_anot=3,t2_int=5,t3_anot=0,t3_int=2,reb_o=1,reb_d=3,ast=7,perd=2,rec=2,tap_f=0,tap_c=0,mates=0,fc=2,fr=3,plus_minus=5,val=16)
calderon_stats_j1 = PlayerGameStats(player=calderon,minutos=21,segundos=30,puntos=10,tl_anot=2,tl_int=2,t2_anot=1,t2_int=3,t3_anot=2,t3_int=4,reb_o=0,reb_d=2,ast=4,perd=1,rec=1,tap_f=0,tap_c=0,mates=0,fc=1,fr=2,plus_minus=6,val=13)
chacho_stats_j1 = PlayerGameStats(player=chacho,minutos=22,segundos=0,puntos=7,tl_anot=1,tl_int=2,t2_anot=3,t2_int=5,t3_anot=0,t3_int=2,reb_o=0,reb_d=2,ast=6,perd=2,rec=1,tap_f=0,tap_c=0,mates=0,fc=1,fr=1,plus_minus=4,val=9)
llull_stats_j1 = PlayerGameStats(player=llull,minutos=22,segundos=0,puntos=11,tl_anot=2,tl_int=2,t2_anot=3,t2_int=6,t3_anot=1,t3_int=3,reb_o=0,reb_d=2,ast=3,perd=1,rec=1,tap_f=0,tap_c=0,mates=0,fc=2,fr=3,plus_minus=8,val=12)
navarro_stats_j1 = PlayerGameStats(player=navarro,minutos=22,segundos=0,puntos=14,tl_anot=4,tl_int=4,t2_anot=2,t2_int=4,t3_anot=2,t3_int=5,reb_o=0,reb_d=1,ast=2,perd=1,rec=1,tap_f=0,tap_c=0,mates=0,fc=1,fr=4,plus_minus=7,val=15)
rudy_stats_j1 = PlayerGameStats(player=rudy,minutos=18,segundos=0,puntos=11,tl_anot=4,tl_int=4,t2_anot=2,t2_int=4,t3_anot=1,t3_int=3,reb_o=1,reb_d=3,ast=2,perd=1,rec=2,tap_f=1,tap_c=0,mates=1,fc=2,fr=3,plus_minus=6,val=16)
ibaka_stats_j1 = PlayerGameStats(player=ibaka,minutos=18,segundos=0,puntos=9,tl_anot=1,tl_int=2,t2_anot=4,t2_int=6,t3_anot=0,t3_int=0,reb_o=2,reb_d=4,ast=0,perd=1,rec=1,tap_f=3,tap_c=0,mates=2,fc=2,fr=2,plus_minus=5,val=15)
garbajosa_stats_j1 = PlayerGameStats(player=garbajosa,minutos=18,segundos=0,puntos=6,tl_anot=0,tl_int=0,t2_anot=3,t2_int=5,t3_anot=0,t3_int=2,reb_o=1,reb_d=3,ast=1,perd=1,rec=0,tap_f=0,tap_c=0,mates=0,fc=2,fr=1,plus_minus=3,val=5)
pau_stats_j1 = PlayerGameStats(player=pau,minutos=18,segundos=0,puntos=16,tl_anot=4,tl_int=5,t2_anot=6,t2_int=10,t3_anot=0,t3_int=0,reb_o=3,reb_d=6,ast=2,perd=2,rec=1,tap_f=2,tap_c=1,mates=1,fc=3,fr=4,plus_minus=9,val=23)
marc_stats_j1 = PlayerGameStats(player=marc,minutos=18,segundos=0,puntos=13,tl_anot=3,tl_int=4,t2_anot=5,t2_int=8,t3_anot=0,t3_int=1,reb_o=2,reb_d=5,ast=3,perd=1,rec=1,tap_f=1,tap_c=1,mates=1,fc=2,fr=3,plus_minus=7,val=19)

team_stats_esp_j1 = TeamGameStats(reb_o=1,reb_d=2,perd=1,rec=0,fc=1)

spain_game_j1 = MyTeamGame(
    team_name="España",
    players_list=[ricky, calderon, chacho, llull, navarro,rudy, ibaka, garbajosa, pau, marc],
    stats_list=[ricky_stats_j1, calderon_stats_j1, chacho_stats_j1, llull_stats_j1, navarro_stats_j1, rudy_stats_j1, ibaka_stats_j1, garbajosa_stats_j1, pau_stats_j1, marc_stats_j1],
    team_stats=team_stats_esp_j1
)

opp_j1_usa = OpponentTeamGame("USA","",200,99,20,25,20,38,13,33,9,35,27,11,3,2,4,10,19,18,-60,116)

j1 = Game(1,datetime.datetime(2008,8,24),"Beijing",spain_game_j1,opp_j1_usa)


# ==========================================================
# JORNADA 2 — ESPAÑA vs FRANCIA
# ==========================================================

ricky_stats_j2 = PlayerGameStats(ricky,23,0,10,2,2,4,7,0,2,1,4,8,2,2,0,0,0,2,3,7,19)
calderon_stats_j2 = PlayerGameStats(calderon,20,0,11,1,1,2,4,2,4,0,2,5,1,1,0,0,0,1,2,5,15)
chacho_stats_j2 = PlayerGameStats(chacho,21,0,6,0,0,3,6,0,2,0,2,6,2,1,0,0,0,1,1,3,8)
llull_stats_j2 = PlayerGameStats(llull,22,0,14,2,2,3,6,2,4,0,2,4,1,1,0,0,0,2,3,6,16)
navarro_stats_j2 = PlayerGameStats(navarro,22,0,17,5,5,3,6,2,5,0,1,3,2,1,0,0,0,2,4,9,16)
rudy_stats_j2 = PlayerGameStats(rudy,18,0,12,3,4,3,5,1,3,1,4,3,1,2,1,0,1,2,3,6,18)
ibaka_stats_j2 = PlayerGameStats(ibaka,18,0,10,2,2,4,6,0,0,2,5,0,1,1,3,0,2,2,2,4,18)
garbajosa_stats_j2 = PlayerGameStats(garbajosa,18,0,4,0,0,2,4,0,2,1,3,1,1,0,0,0,2,1,2,6,5)
pau_stats_j2 = PlayerGameStats(pau,19,0,19,5,6,7,11,0,0,3,7,3,2,1,2,1,1,3,4,10,28)
marc_stats_j2 = PlayerGameStats(marc,19,0,15,3,4,6,9,0,0,2,6,4,1,1,1,1,1,2,3,8,24)

team_stats_esp_j2 = TeamGameStats(reb_o=2,reb_d=3,perd=1,rec=1,fc=2)

spain_game_j2 = MyTeamGame(
    team_name="España",
    players_list=[ricky,calderon,chacho,llull,navarro,rudy,ibaka,garbajosa,pau,marc],
    stats_list=[ricky_stats_j2,calderon_stats_j2,chacho_stats_j2,llull_stats_j2,navarro_stats_j2,rudy_stats_j2,ibaka_stats_j2,garbajosa_stats_j2,pau_stats_j2,marc_stats_j2],
    team_stats=team_stats_esp_j2
)

opp_j2_france = OpponentTeamGame("Francia","",200,88,16,20,18,35,12,28,7,30,24,12,2,3,3,9,17,16,-45,101)

j2 = Game(2,datetime.datetime(2008,8,26),"Beijing",spain_game_j2,opp_j2_france)


# ==========================================================
# JORNADA 3 — ESPAÑA vs CANADÁ
# ==========================================================

ricky_stats_j3 = PlayerGameStats(ricky,24,0,11,3,4,4,6,0,2,1,4,9,2,2,0,0,0,2,4,8,22) #AQUÍ
calderon_stats_j3 = PlayerGameStats(calderon,21,0,8,1,1,2,3,1,3,0,2,6,1,1,0,0,0,1,2,4,14)
chacho_stats_j3 = PlayerGameStats(chacho,20,0,7,1,2,3,5,0,2,0,2,5,2,1,0,0,0,1,1,3,8)
llull_stats_j3 = PlayerGameStats(llull,22,0,13,2,2,4,7,1,3,0,3,3,1,1,0,0,0,2,3,7,15)
navarro_stats_j3 = PlayerGameStats(navarro,21,0,16,4,4,3,6,2,5,0,1,2,2,1,0,0,0,2,4,9,14)
rudy_stats_j3 = PlayerGameStats(rudy,19,0,16,4,4,3,5,2,4,1,5,2,1,2,1,0,1,2,3,8,23)
ibaka_stats_j3 = PlayerGameStats(ibaka,18,0,11,1,2,5,7,0,0,2,6,0,1,1,3,0,2,2,2,5,19)
garbajosa_stats_j3 = PlayerGameStats(garbajosa,17,0,6,0,0,3,5,0,2,1,3,1,1,0,0,0,2,1,3,8,8)
pau_stats_j3 = PlayerGameStats(pau,19,0,22,6,7,8,12,0,0,4,8,3,2,1,2,1,1,3,4,11,33)
marc_stats_j3 = PlayerGameStats(marc,19,0,14,2,3,6,9,0,0,2,7,4,1,1,1,1,1,2,3,7,24)

team_stats_esp_j3 = TeamGameStats(reb_o=1,reb_d=3,perd=2,rec=1,fc=1)

spain_game_j3 = MyTeamGame(
    team_name="España",
    players_list=[ricky,calderon,chacho,llull,navarro,rudy,ibaka,garbajosa,pau,marc],
    stats_list=[ricky_stats_j3,calderon_stats_j3,chacho_stats_j3,llull_stats_j3,navarro_stats_j3,rudy_stats_j3,ibaka_stats_j3,garbajosa_stats_j3,pau_stats_j3,marc_stats_j3],
    team_stats=team_stats_esp_j3
)

j3_opp_canada = OpponentTeamGame("Canada","",200,85,14,18,19,36,11,29,6,28,22,13,2,2,2,8,18,17,-50,90)

j3 = Game(3,datetime.datetime(2008,8,28),"Beijing",spain_game_j3,j3_opp_canada)


# ==========================================================
# CREAR SEASON
# ==========================================================

print("\n")
season1 = Season(competition="Juegos Olimpicos",season="2008",games=[j1,j2,j3])

2026-03-15 20:01:35,713 - INFO - __main__ - Jugador creado: Ricky Rubio (#9)
2026-03-15 20:01:35,715 - INFO - __main__ - Jugador creado: Jose Manuel Calderon (#8)
2026-03-15 20:01:35,715 - INFO - __main__ - Jugador creado: Sergio Rodriguez (#13)
2026-03-15 20:01:35,716 - INFO - __main__ - Jugador creado: Sergio Llull (#23)
2026-03-15 20:01:35,717 - INFO - __main__ - Jugador creado: Juan Carlos Navarro (#7)
2026-03-15 20:01:35,718 - INFO - __main__ - Jugador creado: Rudy Fernandez (#5)
2026-03-15 20:01:35,719 - INFO - __main__ - Jugador creado: Serge Ibaka (#14)
2026-03-15 20:01:35,719 - INFO - __main__ - Jugador creado: Jorge Garbajosa (#15)
2026-03-15 20:01:35,720 - INFO - __main__ - Jugador creado: Pau Gasol (#4)
2026-03-15 20:01:35,721 - INFO - __main__ - Jugador creado: Marc Gasol (#33)
2026-03-15 20:01:35,723 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correctamente para Ricky Rubio (#9)
2026-03-15 20:01:35,724 - INFO - __main__ - 'PlayerGameStats': Stats cargadas correc

In [348]:
# ==========================================================
# EJEMPLOS DE USO
# ==========================================================

#season1.season_boxscore -> # No tiene utilidad visual. Es un dataframe con todas las columnas de "PlayerGameStats"

# ESTADÍSTICAS TOTALES DE LA TEMPORADA
season1.season_totals


,jugador,dorsal,games_played,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
0,Pau Gasol,4,3,56.0,57,15,18,83.33,21,33,...,8,6,3,6,3,3,9,12,30,84
1,Rudy Fernandez,5,3,55.0,39,11,12,91.67,8,14,...,7,3,6,3,0,3,6,9,20,57
2,Juan Carlos Navarro,7,3,65.0,47,13,13,100.0,8,16,...,7,5,3,0,0,0,5,12,25,45
3,Jose Manuel Calderon,8,3,62.5,29,4,4,100.0,5,10,...,15,3,3,0,0,0,3,6,15,42
4,Ricky Rubio,9,3,69.5,29,7,8,87.5,11,18,...,24,6,6,0,0,0,6,10,20,57
5,Sergio Rodriguez,13,3,63.0,20,2,4,50.0,9,16,...,17,6,3,0,0,0,3,3,10,25
6,Serge Ibaka,14,3,54.0,30,4,6,66.67,13,19,...,0,3,3,9,0,6,6,6,14,52
7,Jorge Garbajosa,15,3,53.0,16,0,0,0.0,8,14,...,3,3,0,0,0,4,4,6,17,18
8,Sergio Llull,23,3,66.0,38,6,6,100.0,10,19,...,10,3,3,0,0,0,6,9,21,43
9,Marc Gasol,33,3,56.0,42,8,11,72.73,17,26,...,11,3,3,3,3,3,6,9,22,67


In [349]:
# ==========================================================
# EJEMPLOS DE USO
# ==========================================================

# ESTADÍSTICAS POR PARTIDO DE LA TEMPORADA
season1.season_averages


,jugador,dorsal,games_played,min_played,puntos,tl_anot,tl_int,tl_pct,t2_anot,t2_int,...,ast,perd,rec,tap_f,tap_c,mates,fc,fr,plus_minus,val
4,Pau Gasol,4,3,18.67,19.00,5.00,6.00,83.33,7.00,11.00,...,2.67,2.00,1.0,2.0,1.0,1.00,3.00,4.00,10.00,28.00
6,Rudy Fernandez,5,3,18.33,13.00,3.67,4.00,91.67,2.67,4.67,...,2.33,1.00,2.0,1.0,0.0,1.00,2.00,3.00,6.67,19.00
2,Juan Carlos Navarro,7,3,21.67,15.67,4.33,4.33,100.00,2.67,5.33,...,2.33,1.67,1.0,0.0,0.0,0.00,1.67,4.00,8.33,15.00
1,Jose Manuel Calderon,8,3,20.83,9.67,1.33,1.33,100.00,1.67,3.33,...,5.00,1.00,1.0,0.0,0.0,0.00,1.00,2.00,5.00,14.00
5,Ricky Rubio,9,3,23.17,9.67,2.33,2.67,87.50,3.67,6.00,...,8.00,2.00,2.0,0.0,0.0,0.00,2.00,3.33,6.67,19.00
9,Sergio Rodriguez,13,3,21.00,6.67,0.67,1.33,50.00,3.00,5.33,...,5.67,2.00,1.0,0.0,0.0,0.00,1.00,1.00,3.33,8.33
7,Serge Ibaka,14,3,18.00,10.00,1.33,2.00,66.67,4.33,6.33,...,0.00,1.00,1.0,3.0,0.0,2.00,2.00,2.00,4.67,17.33
0,Jorge Garbajosa,15,3,17.67,5.33,0.00,0.00,0.00,2.67,4.67,...,1.00,1.00,0.0,0.0,0.0,1.33,1.33,2.00,5.67,6.00
8,Sergio Llull,23,3,22.00,12.67,2.00,2.00,100.00,3.33,6.33,...,3.33,1.00,1.0,0.0,0.0,0.00,2.00,3.00,7.00,14.33
3,Marc Gasol,33,3,18.67,14.00,2.67,3.67,72.73,5.67,8.67,...,3.67,1.00,1.0,1.0,1.0,1.00,2.00,3.00,7.33,22.33


In [350]:
# ==========================================================
# EJEMPLOS DE USO
# ==========================================================

# ESTADÍSTICAS AVANZADAS DE LA TEMPORADA - JUGADORES
season1.season_advanced_players


,jugador,dorsal,POSS_USED,OER,EFG%,TOV%,OREB%,FT_RATE,DREB%,TS%,AST/TOV,AST_RATIO%,AST%,STL%,BLK%,USG%,USG_TOT%
0,Pau Gasol,4,46.92,1.21,63.64,10.92,16.74,0.55,32.61,69.65,1.33,14.57,9.56,2.67,11.8,30.37,14.17
1,Rudy Fernandez,5,32.28,1.21,58.33,7.64,5.11,0.5,18.97,66.6,2.33,17.82,7.73,5.44,6.01,21.27,9.75
2,Juan Carlos Navarro,7,41.72,1.13,54.84,10.26,0.0,0.42,4.01,64.0,1.4,14.37,6.7,2.3,0.0,23.26,12.60
3,Jose Manuel Calderon,8,25.76,1.13,59.52,7.36,0.0,0.19,8.35,63.71,5.0,36.8,13.74,2.39,0.0,14.94,7.78
4,Ricky Rubio,9,33.52,0.87,45.83,10.43,4.05,0.33,13.76,52.69,4.0,41.72,19.87,4.3,0.0,17.48,10.12
5,Sergio Rodriguez,13,29.76,0.67,40.91,12.83,0.0,0.18,8.28,42.09,2.83,36.36,15.56,2.37,0.0,17.12,8.99
6,Serge Ibaka,14,24.64,1.22,68.42,12.18,10.42,0.32,24.15,69.32,0.0,0.0,0.0,2.77,18.35,16.54,7.44
7,Jorge Garbajosa,15,23.00,0.7,40.0,11.54,5.31,0.0,14.77,40.0,1.0,11.54,3.32,0.0,0.0,15.73,6.95
8,Sergio Llull,23,34.64,1.1,55.17,6.72,0.0,0.21,9.22,60.05,3.33,22.4,9.22,2.27,0.0,19.02,10.46
9,Marc Gasol,33,34.84,1.21,62.96,6.54,10.04,0.41,27.95,65.95,3.67,24.0,12.27,2.67,5.9,22.55,10.52


In [351]:
# ==========================================================
# EJEMPLOS DE USO
# ==========================================================

# ESTADÍSTICAS AVANZADAS DE LA TEMPORADA - EQUIPO
season1.season_advanced_team

,jugador,POSS,OER,DER,NET,EFG%,TOV%,OREB%,FT_RATE,DREB%,TS%,AST/TOV,AST_RATIO%,AST%,STL%,BLK%,PACE_ACB,PACE_NBA,GAMES_PLAYED,POSS_per_game
0,Totales,296.08,117.2,112.99,4.2,55.4,13.59,27.34,0.33,84.06,60.65,2.27,30.81,40.8,14.54,19.27,89.47,107.36,3,98.69


Comentario final sobre las posesiones:

Si observamos las dos últimas celdas, vemos que la suma de las poseseiones consumidas por los jugadores (∑ POSS_USED = 327.08. Ver fila de "TOTALES" en la celda de "season1.season_advanced_players") NO coincide con las posesiones totales consumidas por el equipo (POSS = 296.08).

Esto no es un error de cálculo... Sino que: ¡Ambas métricas son diferentes!. Que ambos valores tengan los mismos decimales (".08") nos da una pista de esto.

- En las posesiones del equipo (POSS) se restan los rebotes ofensivos.

- Mientras que en las de los jugadores (POSS_USED), se refieren a las "posesiones consumidas", que ellos finalizan. Pueden finalizar con un tiro o una pérdida, pero no con un rebote ofensivo. Es decir, en "POSS_USED", los rebotes ofensivos NO RESTAN.

In [352]:
valor = season1.season_totals.at[11, 'reb_o']
valor

35.0

Si vemos la celda de "season1.season_totals", vemos que el equipo capturó un total de exactamente 35 rebotes ofensivos. De este modo, tenemos que:

∑ POSS_USED = 327.08 (sumatorio extendido a jugadores!)

POSS + OREB = 296.08 + 35 = 331.08

La diferencia de 4 rebotes ofensivos entre POSS_USED y POSS+OREB, se debe a que esos 4 corresponden a rebotes ofensivos de equipo, no asociados a ningún jugador.

...

¡MISTERIO RESUELTO!

---
---

Finalizamos el notebook con varios ejemplos del método "leaderboard", que permite hacer un ranking de los mejores jugadores en función de una estadística avanzada en concreto. Vemos que los jugadores destacados en cada faceta del juego son coherentes con la realidad. Es decir, el ejemplo que hemos propuesto de la selección española, aunque totalmente inventado, es bastante veraz / fidedigno con la historia.

Comenzamos con rankings de estadística convencional:

In [353]:
# ===================================================================================
# EJEMPLOS DE USO - LEADERBOARD_CONV
# ===================================================================================

In [354]:
# RANKING POR: Puntos
season1.leaderboard_conv("puntos",5,False)
# LIDERES -> Pau Gasol & NAVARRO. ¡Nuestro ejemplo es bastante realista!

,jugador,dorsal,puntos
4,Pau Gasol,4,19.00
2,Juan Carlos Navarro,7,15.67
3,Marc Gasol,33,14.00
6,Rudy Fernandez,5,13.00
8,Sergio Llull,23,12.67


In [355]:
# RANKING POR: Asistencias
season1.leaderboard_conv("ast",5,False)
# LIDERES -> Ricky, Chacho y Calde. Bien

,jugador,dorsal,ast
5,Ricky Rubio,9,8.00
9,Sergio Rodriguez,13,5.67
1,Jose Manuel Calderon,8,5.00
3,Marc Gasol,33,3.67
8,Sergio Llull,23,3.33


In [356]:
# RANKING POR: T2%
season1.leaderboard_conv("t2_pct",5,False)
# LIDERES -> Ibaka y Marc. De nuevo, ejemplo consistente.

,jugador,dorsal,t2_pct
7,Serge Ibaka,14,68.42
3,Marc Gasol,33,65.38
4,Pau Gasol,4,63.64
5,Ricky Rubio,9,61.11
6,Rudy Fernandez,5,57.14


In [357]:
# RANKING POR: Faltas recibidas
season1.leaderboard_conv("fr",5,False)
# LIDERES -> Pau y Navarro.

,jugador,dorsal,fr
4,Pau Gasol,4,4.00
2,Juan Carlos Navarro,7,4.00
5,Ricky Rubio,9,3.33
6,Rudy Fernandez,5,3.00
8,Sergio Llull,23,3.00


Finalizamos con rankings de estadística avanzada:

In [358]:
# ===================================================================================
# EJEMPLOS DE USO - LEADERBOARD_ADV
# ===================================================================================

In [359]:
# RANKING POR: POSESIONES CONSUMIDAS
season1.leaderboard_adv("POSS_USED",5,False)
# LIDER -> Pau Gasol. ¡Nuestro ejemplo es bastante realista!

,jugador,dorsal,POSS_USED
0,Pau Gasol,4,46.92
2,Juan Carlos Navarro,7,41.72
9,Marc Gasol,33,34.84
8,Sergio Llull,23,34.64
4,Ricky Rubio,9,33.52


In [360]:
# RANKING POR: OER
season1.leaderboard_adv("OER",5,False)
# LIDER -> Serge Ibaka

,jugador,dorsal,OER
6,Serge Ibaka,14,1.22
0,Pau Gasol,4,1.21
1,Rudy Fernandez,5,1.21
9,Marc Gasol,33,1.21
2,Juan Carlos Navarro,7,1.13


In [361]:
# RANKING POR: USG%
season1.leaderboard_adv("USG%",5,False)
# LIDER -> Pau Gasol


,jugador,dorsal,USG%
0,Pau Gasol,4,30.37
2,Juan Carlos Navarro,7,23.26
9,Marc Gasol,33,22.55
1,Rudy Fernandez,5,21.27
8,Sergio Llull,23,19.02


In [362]:
# RANKING POR: USG_TOT%
season1.leaderboard_adv("USG_TOT%",5,False)
# LIDER -> Pau Gasol

,jugador,dorsal,USG_TOT%
0,Pau Gasol,4,14.17
2,Juan Carlos Navarro,7,12.60
9,Marc Gasol,33,10.52
8,Sergio Llull,23,10.46
4,Ricky Rubio,9,10.12


In [363]:
# RANKING POR: EFG%
season1.leaderboard_adv("EFG%",5,False)
# LIDER -> Serge Ibaka

,jugador,dorsal,EFG%
6,Serge Ibaka,14,68.42
0,Pau Gasol,4,63.64
9,Marc Gasol,33,62.96
3,Jose Manuel Calderon,8,59.52
1,Rudy Fernandez,5,58.33


In [364]:
# RANKING POR: AST%
season1.leaderboard_adv("AST%",5,False)
# LIDER -> Ricky Rubio. El ejemplo construido es bastante coherente

,jugador,dorsal,AST%
4,Ricky Rubio,9,19.87
5,Sergio Rodriguez,13,15.56
3,Jose Manuel Calderon,8,13.74
9,Marc Gasol,33,12.27
0,Pau Gasol,4,9.56


In [365]:
# RANKING POR: AST/TOV
season1.leaderboard_adv("AST/TOV",5,False)
# LIDER -> Jose Manuel Calderon. De nuevo, coherente con un escenario real.

,jugador,dorsal,AST/TOV
3,Jose Manuel Calderon,8,5.0
4,Ricky Rubio,9,4.0
9,Marc Gasol,33,3.67
8,Sergio Llull,23,3.33
5,Sergio Rodriguez,13,2.83


In [366]:
# RANKING POR: STL%
season1.leaderboard_adv("STL%",5,False)
# LIDER -> Rudy Fernandez. El ejemplo construido es bastante coherente

,jugador,dorsal,STL%
1,Rudy Fernandez,5,5.44
4,Ricky Rubio,9,4.3
6,Serge Ibaka,14,2.77
0,Pau Gasol,4,2.67
9,Marc Gasol,33,2.67


Los métodos de la clase season, como el de los leaderboards nos muestran el gran abanico de posibles análisis que podemos hacer. Sería interesante realizar un ejemplo de una temporada con más partidos. Podríamos implementar métodos para ir viendo la variación de las estadísticas a lo largo de la temporada, ver tendencias por partidos, analizar picos de forma, relacionar estadísticas específicas con minutos jugados, o con lugares de juego...

Las posibilidades son infinitas. De momento, en el presente notebook, no vamos a implementar nada más. Con lo que hemos hecho hasta ahora ...

¡Nos damos por ampliamente satisfechos!

Para finalizar, solo resta construir la clase "Calculator", que permite ejecutar la calculadora a partir de un objeto 'season' ya construido.

---
---


**===========================================================**
## **CAPA 3 — APLICACIÓN: CALCULADORA FINAL**
**===========================================================**


Sin más aclaraciones, vamos con el código

In [367]:
# ==========================================================
# CLASE CALCULATOR
# ==========================================================

class Calculator:

    def __init__(self, season):
        self.season = season

    def run(self):

        while True:

            print("=".center(60,"="))
            print("BASKETBALL STATS CALCULATOR (By Pablo Gómez)".center(60,"-"))
            print("MBDA - TFA: 'PROGRAMACIÓN AVANZADA'".center(60,"-"))
            print("=".center(60,"="))
            #print("\n============================================================")
            #print("------BASKETBALL STATS CALCULATOR (By Pablo Gómez)-----\n")
            #print("------MBDA - TFA: 'PROGRAMACIÓN AVANZADA'--------------")
            #print("============================================================\n")
            print("\nBienvenido a la calculadora de estadística.\n")
            print("Ya hemos cargado las estadísticas completas de la temporada.")
            print("A partir de estos datos, vamos a calcular lo que tú quieras.")
            print("Las opciones disponibles son:")

            print("\n------------------------------------------------------------")
            print("0 - SALIR ")
            print("1 - TOTALES de temporada.")
            print("2 - PROMEDIOS de temporada.")
            print("3 - Estadística AVANZADA - JUGADORES.")
            print("4 - Estadística AVANZADA - EQUIPO.")
            print("5 - RANKING de jugadores - Por estadística CONVENCIONAL.")
            print("6 - RANKING de jugadores - Por estadística AVANZADA.")
            print("------------------------------------------------------------\n")

            print("Para continuar, selecciona una opción: 1-6")
            print("No te preocupes. Podrás seleccionar opciones una tras otra, hasta que decidas salir pulsando '0'.\n")

            option = input("\nElige una opción: ")

            if option == "1":
                print("\n")
                print("=".center(80,"="))
                print("TOTALES DE TEMPORADA".center(80,"-"))
                print("=".center(80,"="))
                print("\n")
                print(self.season.season_totals)
                print("\n")

            elif option == "2":
                print("\n")
                print("=".center(80,"="))
                print("PROMEDIOS DE TEMPORADA".center(80,"-"))
                print("=".center(80,"="))
                print("\n")
                print(self.season.season_averages)
                print("\n")

            elif option == "3":
                print("\n")
                print("=".center(80,"="))
                print("ESTADÍSTICA AVANZADA - JUGADORES".center(80,"-"))
                print("=".center(80,"="))
                print("\n")
                print(self.season.season_advanced_players)
                print("\n")

            elif option == "4":
                print("\n")
                print("=".center(80,"="))
                print("ESTADÍSTICA AVANZADA - EQUIPO".center(80,"-"))
                print("=".center(80,"="))
                print("\n")
                print(self.season.season_advanced_team)
                print("\n")

            elif option == "5":
                print("\nLas estadísticas por las que puedes filtrar son:\n")
                print(self.season.season_averages.columns)
                print("\n¡Es importante que escribas bien el nombre de la estadística!")
                print("Ejemplos: puntos, reb_tot, ast.\n")

                stat = input("Selecciona una estadística convencional: ")
                top = int(input("Selecciona cuantos jugadores mostrar: N = "))

                print("\n")
                print("=".center(80,"="))
                print(f"RANKING - STAT CONVENCIONAL: Top {top} jugadores en la estadística '{stat}' ")
                print("=".center(80,"="))
                print("\n")
                print(self.season.leaderboard_conv(stat=stat, top=top))
                print("\n")

            elif option == "6":
                print("\nLas estadísticas por las que puedes filtrar son:\n")
                print(self.season.season_advanced_players.columns)
                print("\n¡Es importante que escribas bien el nombre de la estadística!")
                print("Ejemplos: OER, USG%, TS%\n")

                stat = input("Selecciona una estadística avanzada: ")
                top = int(input("Selecciona cuantos jugadores mostrar: N = "))

                print("\n")
                print("=".center(80,"="))
                print(f"RANKING - STAT AVANZADA: Top {top} jugadores en la estadística '{stat}'")
                print("=".center(80,"="))
                print("\n")
                print(self.season.leaderboard_adv(stat=stat, top=top))
                print("\n")

            elif option == "0":
                print("\n")
                print("=".center(40,"="))
                print("CALCULADORA CERRADA. ¡HASTA PRONTO!".center(40,"-"))
                print("=".center(40,"="))
                print("\n")
                break

            else:
                print("\nOpción no válida. Prueba de nuevo.\n")

### **EJEMPLO FINAL: Clase 'Calculator'**

In [368]:
# Recordemos que el ejemplo de la Selección Española lo habíamos guardado en 'season1'.
# Lo volvemos a ejecugar
season1 = Season(competition="Juegos Olimpicos",season="2008",games=[j1,j2,j3])


2026-03-15 20:02:29,845 - INFO - __main__ - 
'Season' cargada correctamente
Competición: Juegos Olimpicos | Temporada: 2008 | Nº de partidos: 3


In [370]:
# ==========================================================
# EJECUCIÓN DE LA CALCULADORA
# ==========================================================

my_calculator = Calculator(season1)
my_calculator.run()

--------BASKETBALL STATS CALCULATOR (By Pablo Gómez)--------
------------MBDA - TFA: 'PROGRAMACIÓN AVANZADA'-------------

Bienvenido a la calculadora de estadística.

Ya hemos cargado las estadísticas completas de la temporada.
A partir de estos datos, vamos a calcular lo que tú quieras.
Las opciones disponibles son:

------------------------------------------------------------
0 - SALIR 
1 - TOTALES de temporada.
2 - PROMEDIOS de temporada.
3 - Estadística AVANZADA - JUGADORES.
4 - Estadística AVANZADA - EQUIPO.
5 - RANKING de jugadores - Por estadística CONVENCIONAL.
6 - RANKING de jugadores - Por estadística AVANZADA.
------------------------------------------------------------

Para continuar, selecciona una opción: 1-6
No te preocupes. Podrás seleccionar opciones una tras otra, hasta que decidas salir pulsando '0'.


Elige una opción: 1


------------------------------TOTALES DE TEMPORADA------------------------------


                 jugador dorsal games_played min_played punt

---
---
---

## **FINAL DE LA CALCULADORA**

---
---
---
---
---

---
---
---
---
---

## **COMENTARIOS FINALES - MI CONCLUSIÓN SOBRE LA CALCULADORA**

Este nuevo enfoque "orientado a objetos" me parece mucho más chulo, útil, escalable y mantenible. Ya veo una forma de crear un sistema de análisis bastante más robusto, y el usar pandas te abre muchas posibilidades.

---
---
---
---
---

¡Muchas gracias de nuevo!

Un abrazo.

Pablo Gómez.

# **FIN**